In [ ]:
!pip -q install google-genai gradio pandas scikit-learn joblib

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nFILES UPLOADED:")
for name in uploaded:
    print(" -", name)

KeyboardInterrupt: 

In [ ]:
import os

print("CURRENT FILES:")
for f in os.listdir("/content"):
    if f.endswith((".pkl", ".csv")):
        print("✅", f)

In [ ]:
# ============================================================
# FINAL NOXIM ATTACK DETECTION WEB APP
# Random Forest + Gemini Explanation + Gradio
# ============================================================

import pandas as pd
import numpy as np
import joblib
import gradio as gr

from google import genai
from google.colab import userdata


# ============================================================
# 1. LOAD EXISTING MODEL
# ============================================================

MODEL_PATH = "/content/network_attack_detector.pkl"
FEATURE_PATH = "/content/feature_names.pkl"

model = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURE_PATH)

print("✅ Random Forest model loaded")
print("✅ Feature list loaded")
print("Features:", feature_names)


# ============================================================
# 2. CONNECT GEMINI
# ============================================================

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini API connected")


# ============================================================
# 3. GEMINI EXPLANATION
# ============================================================

def explain_attack(filename, row, indicators, confidence):

    feature_text = "\n".join(
        [
            f"- {name}: {value}"
            for name, value in row.items()
        ]
    )

    indicator_text = "\n".join(
        [
            f"- {name}: {importance:.2f}%"
            for name, importance in indicators
        ]
    )

    prompt = f"""
You are a Network-on-Chip cybersecurity analyst.

We have analyzed a Noxim Network-on-Chip simulation using
a Random Forest machine-learning classifier.

File:
{filename}

ML prediction:
ATTACK

Model confidence:
{confidence:.2f}%

Important model indicators:
{indicator_text}

Observed network metrics:
{feature_text}

Explain why this traffic may be suspicious.

Give the explanation in this exact structure:

### Attack Assessment
Explain clearly why the model classified it as an attack.

### Key Evidence
Explain the most important 3 to 5 network indicators and
what their values suggest.

### Network Impact
Explain possible effects such as congestion, packet delivery
degradation, latency, throughput changes, or energy changes.

### Security Interpretation
Explain what kind of suspicious behavior these metrics could
indicate. Do NOT invent a specific attack type unless the
available data supports it.

### Recommended Action
Give 2 or 3 practical defensive actions.

Keep the explanation technical but understandable for a
college project demonstration.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text


# ============================================================
# 4. DETECTION FUNCTION
# ============================================================

def detect_network_attack(file):

    if file is None:
        return (
            "⚠️ Upload a Noxim CSV file.",
            pd.DataFrame(),
            "",
            pd.DataFrame()
        )

    try:

        # Read CSV
        df = pd.read_csv(file)

        # Check required features
        missing = [
            f for f in feature_names
            if f not in df.columns
        ]

        if missing:
            return (
                "❌ Missing model features:\n\n" + "\n".join(missing),
                pd.DataFrame(),
                "",
                df.head(10)
            )

        # Model input
        X = df[feature_names]

        # Prediction
        predictions = model.predict(X)

        probabilities = model.predict_proba(X)

        # ----------------------------------------------------
        # Detection results
        # ----------------------------------------------------

        results = []

        for i, prediction in enumerate(predictions):

            confidence = probabilities[i][prediction] * 100

            status = (
                "🔴 ATTACK"
                if prediction == 1
                else "🟢 NORMAL"
            )

            results.append({
                "Record": i,
                "Prediction": int(prediction),
                "Status": status,
                "Confidence (%)": round(confidence, 2)
            })

        results_df = pd.DataFrame(results)

        attack_count = int((predictions == 1).sum())
        normal_count = int((predictions == 0).sum())

        avg_confidence = float(
            np.max(probabilities, axis=1).mean() * 100
        )

        filename = file.split("/")[-1]

        # ----------------------------------------------------
        # Feature importance
        # ----------------------------------------------------

        importance = model.feature_importances_

        importance_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance (%)": importance * 100
        })

        importance_df = importance_df.sort_values(
            "Importance (%)",
            ascending=False
        )

        top_indicators = []

        for _, r in importance_df.head(5).iterrows():

            top_indicators.append(
                (
                    r["Feature"],
                    r["Importance (%)"]
                )
            )

        indicator_text = "### 🔎 Top Attack Indicators\n\n"

        for feature, importance_value in top_indicators:

            indicator_text += (
                f"**{feature}** — "
                f"{importance_value:.2f}%\n\n"
            )

        # ----------------------------------------------------
        # Summary
        # ----------------------------------------------------

        if attack_count > 0:

            summary = f"""
# 🚨 ATTACK DETECTED

### File
`{filename}`

### Detection
🔴 **ATTACK**

### Suspicious Records
**{attack_count}**

### Normal Records
**{normal_count}**

### Average Confidence
**{avg_confidence:.2f}%**

The Random Forest model detected network traffic
patterns associated with the attack class.
"""

            # ------------------------------------------------
            # Gemini explanation
            # ------------------------------------------------

            # Use the first detected attack record
            attack_index = np.where(predictions == 1)[0][0]

            attack_row = X.iloc[attack_index]

            explanation = explain_attack(
                filename,
                attack_row,
                top_indicators,
                probabilities[attack_index][1] * 100
            )

        else:

            summary = f"""
# 🟢 NORMAL TRAFFIC

### File
`{filename}`

### Detection
🟢 **NORMAL**

### Suspicious Records
**0**

### Normal Records
**{normal_count}**

### Average Confidence
**{avg_confidence:.2f}%**

No attack was detected in the uploaded simulation.
"""

            explanation = """
### 🛡️ AI Security Assessment

The Random Forest model classified the uploaded
simulation as **NORMAL**.

Gemini attack analysis is generated only when
suspicious traffic is detected.
"""

        return (
            summary,
            results_df,
            indicator_text + "\n" + explanation,
            df.head(10)
        )

    except Exception as e:

        return (
            f"# ❌ ERROR\n\n```text\n{str(e)}\n```",
            pd.DataFrame(),
            "",
            pd.DataFrame()
        )


# ============================================================
# 5. GRADIO WEBPAGE
# ============================================================

with gr.Blocks(
    title="Noxim AI Network Attack Detection"
) as demo:

    gr.Markdown(
        """
# 🛡️ Noxim AI Network Attack Detection System

### Machine Learning + Generative AI Security Analysis

Upload a **Noxim Network-on-Chip simulation CSV**.

The system will:

**CSV → Random Forest → Attack Detection → Feature Analysis → Gemini Security Explanation**
"""
    )

    file_input = gr.File(
        label="📁 Upload Noxim CSV",
        file_types=[".csv"],
        type="filepath"
    )

    analyze_button = gr.Button(
        "🔍 ANALYZE NETWORK TRAFFIC",
        variant="primary"
    )

    gr.Markdown("## 📊 Detection Summary")

    summary_output = gr.Markdown()

    gr.Markdown("## 📋 Detection Results")

    results_output = gr.Dataframe(
        interactive=False
    )

    gr.Markdown("## 🧠 AI Security Analysis")

    explanation_output = gr.Markdown()

    gr.Markdown("## 📄 Simulation Data")

    data_output = gr.Dataframe(
        interactive=False
    )

    analyze_button.click(
        fn=detect_network_attack,
        inputs=file_input,
        outputs=[
            summary_output,
            results_output,
            explanation_output,
            data_output
        ]
    )


# ============================================================
# 6. LAUNCH
# ============================================================

print("\n🚀 LAUNCHING WEB APPLICATION...")

demo.launch(
    share=True,
    debug=False
)

In [ ]:
from google.colab import files
import os

print("Please upload 'network_attack_detector.pkl' and 'feature_names.pkl' here:")
uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print("✅", name)

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib
import os

# ============================================================
# 1. LOAD THE MULTI-CLASS MODEL & FEATURES
# ============================================================

MODEL_PATH = "multiclass_noc_attack_detector.pkl"
FEATURE_PATH = "multiclass_noc_feature_names.pkl"

if not os.path.exists(MODEL_PATH) or not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(
        "Multi-class model or features file not found. Please ensure they are uploaded or trained."
    )

model = joblib.load(MODEL_PATH)
features = joblib.load(FEATURE_PATH)

print("✅ Multi-class Model Loaded:", type(model).__name__)
print("✅ Features Expected:", features)
print("✅ Classes:", list(model.classes_))

# ============================================================
# 2. ANALYSIS PIPELINE FOR GRADIO
# ============================================================

def analyze_noc_simulation(file):
    if file is None:
        return "⚠️ Please upload a Noxim CSV file.", "", ""

    try:
        # Read uploaded simulation
        df = pd.read_csv(file)

        # Validate features
        missing = [f for f in features if f not in df.columns]
        if missing:
            return f"❌ Missing expected features: {missing}", "", ""

        # Select inputs
        X = df[features].copy()
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

        # Get predictions and probabilities
        pred = model.predict(X)[0]
        probs = model.predict_proba(X)[0]
        confidence = max(probs) * 100

        # 1. Prediction Status Banner
        status_color = "#22c55e" if pred == "NORMAL" else "#ef4444"
        status_html = f"""
        <div style="background: #0f172a; color: white; border-radius: 12px; padding: 24px;">
            <div style="font-size: 11px; color: #94a3b8; letter-spacing: 1.5px; font-weight: bold;">
                CLASSIFIER STATUS
            </div>
            <div style="font-size: 28px; font-weight: 800; margin-top: 8px; color: {status_color};">
                {pred}
            </div>
            <div style="margin-top: 8px; font-size: 14px; color: #cbd5e1;">
                Confidence: <b>{confidence:.2f}%</b>
            </div>
        </div>
        """

        # 2. Class Probabilities Progress Bars
        prob_html = """
        <div style="background: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 18px;">
            <h4 style='margin-top:0;'>Confidence Breakdown</h4>
        """
        for cls, p in zip(model.classes_, probs):
            pct = p * 100
            prob_html += f"""
            <div style="margin-top: 12px;">
                <div style="display: flex; justify-content: space-between; font-size: 12px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                    <span>{cls}</span>
                    <span>{pct:.2f}%</span>
                </div>
                <div style="height: 8px; background: #f1f5f9; border-radius: 99px; overflow: hidden;">
                    <div style="width: {pct}%; height: 100%; background: #3b82f6; border-radius: 99px;"></div>
                </div>
            </div>
            """
        prob_html += "</div>"

        # 3. Features Metrics Table
        row = X.iloc[0]
        metrics_html = """
        <div style="background: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 18px;">
            <h4 style='margin-top:0;'>Network Metrics Evidence</h4>
            <table style='width: 100%; font-size: 12px; border-collapse: collapse;'>
                <thead>
                    <tr style='border-bottom: 2px solid #e2e8f0; text-align: left;'>
                        <th style='padding: 6px 0; color: #475569;'>Feature</th>
                        <th style='padding: 6px 0; color: #475569; text-align: right;'>Value</th>
                    </tr>
                </thead>
                <tbody>
        """
        for f in features:
            val = float(row[f])
            metrics_html += f"""
            <tr style='border-bottom: 1px solid #f1f5f9;'>
                <td style='padding: 8px 0; font-family: monospace; color: #334155;'>{f}</td>
                <td style='padding: 8px 0; text-align: right; font-weight: bold; color: #0f172a;'>{val:.6f}</td>
            </tr>
            """
        metrics_html += "</tbody></table></div>"

        return status_html, prob_html, metrics_html

    except Exception as e:
        return f"❌ Error analyzing file: {str(e)}", "", ""

# ============================================================
# 3. BUILD GRADIO INTERFACE
# ============================================================

try:
    demo.close()
except:
    pass

with gr.Blocks(title="Noxim Multi-Class Security Dashboard") as demo:
    gr.Markdown("""
    # 🛡️ Noxim Network-on-Chip Multi-Class Security System
    Analyze simulation parameters to classify traffic as **NORMAL**, **DOS_FLOODING**, **BLACKHOLE**, or **ROUTING_MANIPULATION**.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label="📁 Upload Noxim CSV Simulation file",
                file_types=[".csv"],
                type="filepath"
            )
            analyze_btn = gr.Button("🔍 Analyze Traffic Pattern", variant="primary")

        with gr.Column(scale=2):
            output_status = gr.HTML()
            output_probs = gr.HTML()
            output_metrics = gr.HTML()

    analyze_btn.click(
        fn=analyze_noc_simulation,
        inputs=file_input,
        outputs=[output_status, output_probs, output_metrics]
    )

# Launch application
demo.launch(share=True)

In [ ]:
import os
print("Model exists:", os.path.exists("/content/network_attack_detector.pkl"))
print("Features exist:", os.path.exists("/content/feature_names.pkl"))

In [ ]:
# ============================================================
# NOXIM AI NETWORK ATTACK DETECTION
# FINAL DEMO
# CSV -> RANDOM FOREST -> GEMINI -> GRADIO WEBSITE
# ============================================================

import os
import glob
import pandas as pd
import numpy as np
import gradio as gr

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from google import genai
from google.colab import userdata


# ============================================================
# 1. FIND UPLOADED NOXIM CSV FILES
# ============================================================

csv_files = []

for f in glob.glob("/content/*.csv"):

    name = os.path.basename(f).lower()

    if name.startswith("attack_") or name.startswith("normal_"):
        csv_files.append(f)

print("CSV FILES FOUND:", len(csv_files))

if len(csv_files) < 4:
    raise Exception(
        "Not enough attack/normal CSV files found in /content."
    )

for f in sorted(csv_files):
    print(" -", os.path.basename(f))


# ============================================================
# 2. LOAD DATA
# ============================================================

dataframes = []

for file in csv_files:

    df = pd.read_csv(file)

    filename = os.path.basename(file).lower()

    if filename.startswith("attack_"):
        df["label"] = 1

    elif filename.startswith("normal_"):
        df["label"] = 0

    dataframes.append(df)


data = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nDATASET SHAPE:", data.shape)

print("\nCLASS DISTRIBUTION:")
print(data["label"].value_counts())


# ============================================================
# 3. SELECT NUMERIC FEATURES
# ============================================================

X = data.drop(columns=["label"])

X = X.select_dtypes(
    include=["number"]
)

# Remove constant columns
X = X.loc[
    :,
    X.nunique() > 1
]

y = data["label"]

feature_names = X.columns.tolist()

print("\nFEATURES USED:")

for f in feature_names:
    print(" -", f)

print("\nNUMBER OF FEATURES:", len(feature_names))


# ============================================================
# 4. TRAIN RANDOM FOREST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

print("\nTraining Random Forest...")

model.fit(
    X_train,
    y_train
)

y_test_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(
    f"\nMODEL TEST ACCURACY: {accuracy * 100:.2f}%"
)


# ============================================================
# 5. GEMINI
# ============================================================

try:

    GEMINI_API_KEY = userdata.get(
        "GEMINI_API_KEY"
    )

    if GEMINI_API_KEY:

        client = genai.Client(
            api_key=GEMINI_API_KEY
        )

        GEMINI_READY = True

        print("✅ Gemini API connected")

    else:

        GEMINI_READY = False

        print(
            "⚠️ GEMINI_API_KEY not found."
        )

except Exception as e:

    GEMINI_READY = False

    print(
        "⚠️ Gemini connection failed:",
        e
    )


# ============================================================
# 6. GEMINI SECURITY EXPLANATION
# ============================================================

def generate_gemini_explanation(
    filename,
    attack_row,
    top_indicators,
    confidence
):

    if not GEMINI_READY:

        return """
### ⚠️ Gemini Not Connected

The Random Forest successfully detected the attack,
but the Gemini API key is not currently available.

Add `GEMINI_API_KEY` to Colab Secrets to enable
the AI security explanation.
"""


    metrics_text = "\n".join(
        [
            f"{feature}: {value}"
            for feature, value
            in attack_row.items()
        ]
    )


    indicators_text = "\n".join(
        [
            f"{feature}: {importance:.2f}%"
            for feature, importance
            in top_indicators
        ]
    )


    prompt = f"""
You are an expert Network-on-Chip cybersecurity analyst.

Analyze this Noxim simulation.

FILE:
{filename}

RANDOM FOREST RESULT:
ATTACK

MODEL CONFIDENCE:
{confidence:.2f}%

TOP MACHINE LEARNING INDICATORS:
{indicators_text}

OBSERVED NETWORK METRICS:
{metrics_text}

Explain the detection for a college project demonstration.

Use these sections:

### 🚨 Attack Assessment

Explain why the machine-learning model classified
the traffic as suspicious.

### 🔎 Key Evidence

Explain the most important network metrics and
their observed values.

Focus on:
- throughput
- received flits
- received packets
- delay
- energy
- packet delivery / ideal flit ratio

### 🌐 Network Impact

Explain possible effects on:
- congestion
- latency
- throughput
- packet delivery
- energy consumption

### 🛡️ Security Interpretation

Explain what suspicious network behaviour these
metrics may indicate.

IMPORTANT:
Do NOT claim a specific attack such as DoS, DDoS,
spoofing, flooding, or routing attack unless the
available Noxim data actually proves it.

### ✅ Recommended Actions

Give 3 practical defensive actions.

Keep it technical but easy to understand.
"""


    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:

        return f"""
### ⚠️ Gemini Error

```text
{str(e)}
```
"""


In [ ]:
import gradio as gr

# Define the demo interface before launching
with gr.Blocks() as demo:
    gr.Markdown("# 🛡️ Noxim AI Network Attack Detection System")
    gr.Markdown("The Gradio UI is ready.")

demo.launch(
    share=True,
    debug=False
)

In [ ]:
print("demo exists:", "demo" in globals())

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib

MODEL_PATH = "/content/network_attack_detector.pkl"
FEATURE_PATH = "/content/feature_names.pkl"

# Save the in-memory model and features to disk first
joblib.dump(model, MODEL_PATH)
joblib.dump(feature_names, FEATURE_PATH)

# Load the files
model = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURE_PATH)

def detect_attack(file):

    if file is None:
        return "Please upload a CSV.", None, "", None

    df = pd.read_csv(file)

    missing = [f for f in feature_names if f not in df.columns]

    if missing:
        return f"Missing features: {missing}", None, "", df.head(10)

    X = df[feature_names]

    prediction = model.predict(X)
    probability = model.predict_proba(X)

    results = []

    for i, pred in enumerate(prediction):
        results.append({
            "Record": i,
            "Prediction": int(pred),
            "Status": "🔴 ATTACK" if pred == 1 else "🟢 NORMAL",
            "Confidence (%)": round(probability[i][pred] * 100, 2)
        })

    results_df = pd.DataFrame(results)

    attacks = int((prediction == 1).sum())
    normal = int((prediction == 0).sum())
    confidence = float(np.max(probability, axis=1).mean() * 100)

    if attacks:
        summary = f"""
# 🚨 ATTACK DETECTED

**Suspicious Records:** {attacks}

**Normal Records:** {normal}

**Average Confidence:** {confidence:.2f}%

The Random Forest model detected suspicious
network traffic.
"""
    else:
        summary = f"""
# 🟢 NORMAL TRAFFIC

**Suspicious Records:** 0

**Normal Records:** {normal}

**Average Confidence:** {confidence:.2f}%

No attack was detected.
"""

    importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance (%)": model.feature_importances_ * 100
    }).sort_values("Importance (%)", ascending=False)

    explanation = "## 🔎 TOP ATTACK INDICATORS\n\n"

    for _, row in importance.iterrows():
        explanation += (
            f"**{row['Feature']}** — "
            f"{row['Importance (%)']:.2f}%\n\n"
        )

    return summary, results_df, explanation, df.head(10)


with gr.Blocks(title="Network Attack Detection") as demo:

    gr.Markdown("""
# 🛡️ Network Attack Detection System

### Noxim Network-on-Chip Security Analysis

Upload a Noxim simulation CSV to detect **NORMAL** or **ATTACK** traffic.
""")

    file_input = gr.File(
        label="📁 Upload Noxim CSV",
        file_types=[".csv"],
        type="filepath"
    )

    button = gr.Button(
        "🔍 ANALYZE NETWORK TRAFFIC",
        variant="primary"
    )

    summary = gr.Markdown()

    results = gr.Dataframe(
        label="Detection Results",
        interactive=False
    )

    explanation = gr.Markdown()

    data = gr.Dataframe(
        label="Simulation Data",
        interactive=False
    )

    button.click(
        detect_attack,
        inputs=file_input,
        outputs=[summary, results, explanation, data]
    )

print("✅ Gradio interface created successfully")

In [ ]:
demo.launch(share=True, debug=False)

In [ ]:
print("DEMO:", demo)

In [ ]:
demo.launch(
    share=True,
    debug=True,
    inbrowser=True
)

In [ ]:
demo.launch(
    share=True,
    debug=False,
    inbrowser=True
)

In [ ]:
demo.launch(
    share=True,
    debug=True
)

In [ ]:
import os
from google.colab import files

model_path = "/content/network_attack_detector.pkl"
features_path = "/content/feature_names.pkl"

if os.path.exists(model_path) and os.path.exists(features_path):
    files.download(model_path)
    files.download(features_path)
else:
    print("Error: Files not found. Please run the cell that trains and saves the model first.")

In [ ]:
import os
import joblib

print("Checking current Colab runtime...")

print("\nFiles in /content:")
print(os.listdir("/content")[:50])

print("\nVariables available:")
print([x for x in ["model", "rf_model", "clf", "feature_names", "features"] if x in globals()])

In [ ]:
if 'model' in globals() and 'feature_names' in globals():
    joblib.dump(model, "/content/network_attack_detector.pkl")
    joblib.dump(feature_names, "/content/feature_names.pkl")

    print("✅ Model saved")
    print("✅ Feature names saved")
else:
    print("Error: 'model' or 'feature_names' is not defined. Please rerun the model training cell first.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print("Drive connected")
print(os.listdir("/content/drive/MyDrive")[:50])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print("🔎 Searching Google Drive for network CSV files...\n")

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.lower().endswith(".csv"):
            matches.append(os.path.join(root, file))

print(f"Found {len(matches)} CSV files.\n")

for path in matches[:100]:
    print(path)

In [ ]:
os.walk("/content/drive/MyDrive")

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print("✅", name)

In [ ]:
# ============================================================
# YATRA - NETWORK ATTACK DETECTION
# CLEAN TRAINING PIPELINE
# ============================================================

import os
import glob
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("🔎 Searching uploaded CSV files...")

# ------------------------------------------------------------
# 1. Find uploaded CSV files
# ------------------------------------------------------------

csv_files = glob.glob("/content/*.csv")

print(f"Found {len(csv_files)} CSV files")

if len(csv_files) == 0:
    raise FileNotFoundError("❌ No CSV files found in /content")

# ------------------------------------------------------------
# 2. Load and label files
# ------------------------------------------------------------

frames = []

for path in csv_files:

    filename = os.path.basename(path).lower()

    try:
        df = pd.read_csv(path)

        # Determine class from filename
        if filename.startswith("attack"):
            df["target"] = 1
        elif filename.startswith("normal"):
            df["target"] = 0
        else:
            print("⚠️ Skipping unknown file:", filename)
            continue

        df["source_file"] = filename
        frames.append(df)

        print(f"✅ {filename:25s} {df.shape}")

    except Exception as e:
        print(f"❌ Could not read {filename}: {e}")

# ------------------------------------------------------------
# 3. Combine everything
# ------------------------------------------------------------

data = pd.concat(frames, ignore_index=True)

print("\n===================================")
print("COMBINED DATASET")
print("===================================")

print("Shape:", data.shape)
print("\nClass distribution:")
print(data["target"].value_counts())

# ------------------------------------------------------------
# 4. Select numeric ML features
# ------------------------------------------------------------

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()

# target must NOT be used as input
if "target" in numeric_cols:
    numeric_cols.remove("target")

# Remove obvious ID/index columns if present
remove_cols = [
    "id",
    "index",
    "record",
    "record_id"
]

feature_cols = [
    c for c in numeric_cols
    if c.lower() not in remove_cols
]

print("\n===================================")
print("FEATURES")
print("===================================")

print("Number of features:", len(feature_cols))

for col in feature_cols:
    print(" -", col)

# ------------------------------------------------------------
# 5. Prepare X and y
# ------------------------------------------------------------

X = data[feature_cols].copy()
y = data["target"].copy()

# Replace infinity / missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# ------------------------------------------------------------
# 6. Train/test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))

# ------------------------------------------------------------
# 7. Random Forest
# ------------------------------------------------------------

print("\n🌲 Training Random Forest...")

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

# ------------------------------------------------------------
# 8. Prediction
# ------------------------------------------------------------

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n===================================")
print("MODEL RESULTS")
print("===================================")

print(f"🎯 Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["NORMAL", "ATTACK"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ------------------------------------------------------------
# 9. Feature importance
# ------------------------------------------------------------

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\n===================================")
print("TOP ATTACK INDICATORS")
print("===================================")

for _, row in importance.head(10).iterrows():
    print(
        f"{row['feature']} "
        f"→ {row['importance'] * 100:.2f}%"
    )

# ------------------------------------------------------------
# 10. Save model
# ------------------------------------------------------------

MODEL_PATH = "/content/network_attack_detector.pkl"
FEATURE_PATH = "/content/feature_names.pkl"

joblib.dump(model, MODEL_PATH)
joblib.dump(feature_cols, FEATURE_PATH)

print("\n===================================")
print("SAVED")
print("===================================")

print("✅", MODEL_PATH)
print("✅", FEATURE_PATH)

print("\n🚀 RANDOM FOREST PIPELINE COMPLETE")

In [ ]:
# ============================================================
# YATRA — DATASET AUDIT
# ============================================================

import glob
import os
import pandas as pd

print("========== YATRA DATA AUDIT ==========\n")

files = sorted(glob.glob("/content/*.csv"))

print("CSV FILES FOUND:", len(files))
print()

total_rows = 0

for path in files:
    try:
        df = pd.read_csv(path)

        print(
            f"{os.path.basename(path):25s} "
            f"rows={len(df):6d}  "
            f"columns={len(df.columns):3d}"
        )

        total_rows += len(df)

    except Exception as e:
        print("ERROR:", os.path.basename(path), e)

print("\n--------------------------------------")
print("TOTAL CSV ROWS:", total_rows)

# Check combined dataset if it still exists
if "data" in globals():

    print("\nCOMBINED DATASET:", data.shape)

    print("\nTARGET DISTRIBUTION:")
    print(data["target"].value_counts())

    print("\nTARGET PERCENTAGE:")
    print(
        data["target"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print("\nDUPLICATE ROWS:", data.duplicated().sum())

    print("\nFIRST 5 ROWS:")
    display(data.head())

else:
    print("\n⚠️ 'data' variable is not available.")

print("\n======================================")

In [ ]:
# ============================================================
# YATRA — FINAL RANDOM FOREST VALIDATION + MODEL SAVE
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

print("==============================================")
print("       YATRA NETWORK ATTACK DETECTION")
print("==============================================")

# ------------------------------------------------------------
# 1. Check dataset
# ------------------------------------------------------------

print("\nDataset shape:", data.shape)

print("\nClass distribution:")
print(data["target"].value_counts())

# ------------------------------------------------------------
# 2. Get the exact features used by our model
# ------------------------------------------------------------

if "feature_names" in globals():

    FEATURES = feature_names

else:

    # fallback: automatically select numeric columns
    FEATURES = [
        c for c in data.columns
        if c != "target"
        and pd.api.types.is_numeric_dtype(data[c])
    ]

print("\nFeatures used:")
for f in FEATURES:
    print(" -", f)

print("\nNumber of features:", len(FEATURES))

# ------------------------------------------------------------
# 3. Prepare X and y
# ------------------------------------------------------------

X = data[FEATURES].copy()
y = data["target"].astype(int)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

# ------------------------------------------------------------
# 4. Create Random Forest
# ------------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# ------------------------------------------------------------
# 5. 5-FOLD STRATIFIED CROSS VALIDATION
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("\n==============================================")
print("5-FOLD CROSS VALIDATION")
print("==============================================")

for i, score in enumerate(scores, 1):
    print(f"Fold {i}: {score * 100:.2f}%")

print("----------------------------------------------")
print(f"Mean Accuracy: {scores.mean() * 100:.2f}%")
print(f"Std Deviation: {scores.std() * 100:.2f}%")

# ------------------------------------------------------------
# 6. Train FINAL model on ALL available data
# ------------------------------------------------------------

rf.fit(X, y)

print("\n==============================================")
print("FINAL MODEL")
print("==============================================")

print("Random Forest trained on all", len(data), "records.")

# ------------------------------------------------------------
# 7. Feature importance
# ------------------------------------------------------------

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTOP ATTACK INDICATORS")
print("----------------------------------------------")

for _, row in importance.iterrows():
    print(
        f"{row['feature']}: "
        f"{row['importance'] * 100:.2f}%"
    )

# ------------------------------------------------------------
# 8. Save model + feature names
# ------------------------------------------------------------

SAVE_DIR = "/content/drive/MyDrive/NetworkAttackDetection"

os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    SAVE_DIR,
    "network_attack_detector.pkl"
)

FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "feature_names.pkl"
)

joblib.dump(rf, MODEL_PATH)
joblib.dump(FEATURES, FEATURE_PATH)

print("\n==============================================")
print("MODEL SAVED")
print("==============================================")

print("Model:")
print(MODEL_PATH)

print("\nFeatures:")
print(FEATURE_PATH)

print("\n==============================================")
print("PIPELINE READY")
print("==============================================")

In [ ]:
# ============================================================
# YATRA — LIVE NETWORK ATTACK PREDICTION
# ============================================================

import pandas as pd
import numpy as np

print("==============================================")
print("       YATRA LIVE ATTACK DETECTOR")
print("==============================================")

# Make sure final model exists
if "rf" not in globals():
    raise RuntimeError("Random Forest model not found. Run the previous cell first.")

# ------------------------------------------------------------
# Use one real record from the dataset as a demo
# ------------------------------------------------------------

sample = data[FEATURES].iloc[[0]].copy()

print("\nAnalyzing network traffic record...")
print("Features:", sample.shape)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

prediction = rf.predict(sample)[0]

probabilities = rf.predict_proba(sample)[0]

# Find probability corresponding to predicted class
class_index = list(rf.classes_).index(prediction)

confidence = probabilities[class_index] * 100

# ------------------------------------------------------------
# Convert prediction to readable label
# ------------------------------------------------------------

if int(prediction) == 1:
    status = "ATTACK"
else:
    status = "NORMAL"

# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf.feature_importances_,
    "value": sample.iloc[0].values
})

importance["impact"] = (
    importance["importance"] *
    importance["value"].abs()
)

importance = importance.sort_values(
    "impact",
    ascending=False
)

# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n==============================================")
print("DETECTION RESULT")
print("==============================================")

print("Prediction :", status)
print(f"Confidence : {confidence:.2f}%")

print("\nCLASS PROBABILITIES")

for cls, prob in zip(rf.classes_, probabilities):

    label = "ATTACK" if int(cls) == 1 else "NORMAL"

    print(
        f"{label:8s}: {prob * 100:.2f}%"
    )

print("\nTOP NETWORK INDICATORS")
print("----------------------------------------------")

for _, row in importance.head(5).iterrows():

    print(
        f"{row['feature']} "
        f"| value={row['value']:.4f} "
        f"| importance={row['importance'] * 100:.2f}%"
    )

print("\n==============================================")
print("LIVE DETECTION COMPLETE")
print("==============================================")

In [ ]:
# ============================================================
# YATRA — GEMINI EXPLAINABLE AI
# ============================================================

import os
from google import genai

# ------------------------------------------------------------
# Gemini client
# ------------------------------------------------------------

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

# ------------------------------------------------------------
# Build attack-indicator information
# ------------------------------------------------------------

top_indicators = importance.head(5)

indicator_text = "\n".join([
    f"- {row['feature']}: value={row['value']:.4f}, "
    f"model_importance={row['importance']*100:.2f}%"
    for _, row in top_indicators.iterrows()
])

# ------------------------------------------------------------
# Gemini prompt
# ------------------------------------------------------------

prompt = f"""
You are the Explainable AI security analyst inside YATRA.

Analyze this Network-on-Chip traffic detection result.

Prediction: {status}
Confidence: {confidence:.2f}%

Top network indicators:
{indicator_text}

Explain the result for an engineering student.

Give exactly these sections:

1. Detection
2. Why it was classified this way
3. Important network indicators
4. Security interpretation
5. Recommended action

Important:
- Do not invent network measurements.
- Only use the supplied prediction and indicators.
- Clearly state that the Random Forest model made the classification.
- Do not claim that feature importance proves causation.
"""

# ------------------------------------------------------------
# Generate explanation
# ------------------------------------------------------------

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

explanation = response.text

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("==============================================")
print("        YATRA — AI SECURITY EXPLANATION")
print("==============================================\n")

print(explanation)

print("\n==============================================")
print("GEMINI EXPLANATION COMPLETE")
print("==============================================")

In [ ]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

print("Gemini key loaded:", bool(key))
print("Key length:", len(key) if key else 0)

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini client initialized successfully")

In [ ]:
# TEST GEMINI EXPLANATION

attack_type = "ATTACK"
confidence = 100.0

top_indicators = [
    "total_received_flits",
    "total_received_packets",
    "max_delay_cycles",
    "global_average_delay_cycles",
    "network_throughput_flits_per_cycle"
]

prompt = f"""
You are an explainable AI cybersecurity analyst.

A network traffic detection model classified this traffic as:
ATTACK

Confidence:
{confidence}%

Top indicators responsible for the prediction:
{top_indicators}

Explain:
1. Why this traffic may be classified as an attack.
2. What the important indicators suggest.
3. What a security administrator should do next.

Keep the explanation concise and suitable for a cybersecurity dashboard.
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("========== GEMINI EXPLANATION ==========")
print(response.text)

In [ ]:
# FINAL YATRA RESULT

yatra_result = {
    "prediction": attack_type,
    "confidence": confidence,
    "top_indicators": top_indicators,
    "explanation": response.text
}

print("========== YATRA FINAL RESULT ==========")
print(yatra_result)

In [ ]:
import glob, os, pandas as pd

files = sorted(glob.glob("/content/*.csv"))

print("TOTAL FILES:", len(files))

for f in files:
    df = pd.read_csv(f)
    print("\nFILE:", os.path.basename(f))
    print("ROWS:", len(df))
    print("COLUMNS:", list(df.columns))

    for c in df.columns:
        if any(x in c.lower() for x in ["attack","type","label","class","scenario"]):
            print("  POSSIBLE LABEL:", c, "=>", df[c].unique()[:20])

In [ ]:
import glob, os, pandas as pd

files = sorted(glob.glob("/content/*.csv"))

rows = []

for f in files:
    name = os.path.basename(f)

    if name.startswith("normal"):
        attack_type = "NORMAL"
    elif name == "attack_node27.csv":
        attack_type = "NODE-27 ATTACK"
    elif name.startswith("attack_"):
        attack_type = "NETWORK ATTACK"
    else:
        attack_type = "UNKNOWN"

    rows.append({
        "File": name,
        "Attack_Type": attack_type
    })

attack_table = pd.DataFrame(rows)

display(attack_table)

In [ ]:
# ============================================================
# MULTI-CLASS NETWORK ATTACK MODEL
# ============================================================

import glob, os
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

files = sorted(glob.glob("/content/*.csv"))

frames = []

for f in files:
    name = os.path.basename(f).lower()
    df = pd.read_csv(f)

    if name.startswith("normal"):
        label = "NORMAL"
    elif name == "attack_node27.csv":
        label = "NODE-27 ATTACK"
    elif name.startswith("attack_"):
        label = "NETWORK ATTACK"
    else:
        continue

    df["attack_type"] = label
    frames.append(df)

data_multi = pd.concat(frames, ignore_index=True)

print("DATASET:", data_multi.shape)
print("\nCLASSES:")
print(data_multi["attack_type"].value_counts())

# Numeric features only
features = [
    c for c in data_multi.select_dtypes(include=np.number).columns
    if c != "attack_type"
]

# Remove constant columns
features = [
    c for c in features
    if data_multi[c].nunique() > 1
]

X = data_multi[features].copy()
y = data_multi["attack_type"]

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

print("\nFEATURES:", len(features))

# ------------------------------------------------------------
# Cross-validation
# ------------------------------------------------------------

model_multi = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model_multi,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("\n5-FOLD ACCURACY:")
for i, score in enumerate(scores, 1):
    print(f"Fold {i}: {score*100:.2f}%")

print(f"\nMean Accuracy: {scores.mean()*100:.2f}%")

# ------------------------------------------------------------
# Final model
# ------------------------------------------------------------

model_multi.fit(X, y)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

joblib.dump(
    model_multi,
    "/content/multiclass_network_attack_detector.pkl"
)

joblib.dump(
    features,
    "/content/multiclass_feature_names.pkl"
)

print("\n====================================")
print("MULTI-CLASS MODEL READY")
print("====================================")
print("Classes:", list(model_multi.classes_))
print("Model saved.")

In [ ]:
# ============================================================
# STEP 2 — TEST MULTI-CLASS MODEL
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os

# Load model + features
model = joblib.load("/content/multiclass_network_attack_detector.pkl")
features = joblib.load("/content/multiclass_feature_names.pkl")

# Test files
test_files = [
    "/content/normal_1.csv",
    "/content/attack_101.csv",
    "/content/attack_node27.csv"
]

for file in test_files:

    df = pd.read_csv(file)

    X_test = df[features].copy()
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.fillna(X_test.median())

    # Prediction
    prediction = model.predict(X_test)[0]

    # Probabilities
    probabilities = model.predict_proba(X_test)[0]

    print("\n" + "="*60)
    print("FILE:", os.path.basename(file))
    print("PREDICTION:", prediction)

    print("\nCLASS CONFIDENCE:")
    for cls, prob in zip(model.classes_, probabilities):
        print(f"{cls}: {prob*100:.2f}%")

    # Feature importance
    importance = pd.DataFrame({
        "Feature": features,
        "Importance": model.feature_importances_
    }).sort_values("Importance", ascending=False)

    print("\nTOP INDICATORS:")
    print(importance.head(5).to_string(index=False))

In [ ]:
# ============================================================
# STEP 3 — AI SECURITY EXPLANATION WITH GEMINI
# ============================================================

import pandas as pd
import numpy as np
import joblib
from google import genai
from google.colab import userdata

# Load model
model = joblib.load("/content/multiclass_network_attack_detector.pkl")
features = joblib.load("/content/multiclass_feature_names.pkl")

# Gemini
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

# ------------------------------------------------------------
# CHANGE THIS FILE TO TEST
# ------------------------------------------------------------
file = "/content/attack_101.csv"

df = pd.read_csv(file)

X = df[features].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Prediction
prediction = model.predict(X)[0]

# Probability
probabilities = model.predict_proba(X)[0]
confidence = max(probabilities) * 100

# ------------------------------------------------------------
# Collect network metrics
# ------------------------------------------------------------

metrics = {}

for col in features:
    metrics[col] = float(X.iloc[0][col])

# ------------------------------------------------------------
# Gemini prompt
# ------------------------------------------------------------

prompt = f"""
You are a Network-on-Chip security analyst.

Analyze the following ML detection result.

Predicted class:
{prediction}

ML confidence:
{confidence:.2f}%

Network metrics:
{metrics}

Important:
- Do NOT invent an attack type that is not supported by the ML label.
- "NETWORK ATTACK" means the model detected the attack scenario represented by that dataset.
- "NODE-27 ATTACK" means the dataset represents an attack involving node 27.
- Explain the evidence using the actual metrics.
- Do not claim that the attack is specifically DDoS, DoS, flooding, etc. unless the provided label supports it.

Return exactly these sections:

1. Detection
2. Confidence
3. Key Indicators
4. Why This Was Detected
5. Security Impact
6. Recommended Action

Keep the explanation concise and suitable for a cybersecurity dashboard.
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# ------------------------------------------------------------
# DISPLAY RESULT
# ------------------------------------------------------------

print("=" * 70)
print("NETWORK SECURITY AI ANALYSIS")
print("=" * 70)

print(f"\nPrediction : {prediction}")
print(f"Confidence : {confidence:.2f}%")

print("\nAI EXPLANATION:")
print(response.text)

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# STEP 3 — AI SECURITY EXPLANATION WITH GEMINI
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os
from google import genai
from google.colab import userdata

MODEL_PATH = "/content/multiclass_network_attack_detector.pkl"
FEATURE_PATH = "/content/multiclass_feature_names.pkl"

if not os.path.exists(MODEL_PATH) or not os.path.exists(FEATURE_PATH):
    print("❌ Error: Multi-class model files not found.")
    print("Please scroll up and run the 'MULTI-CLASS NETWORK ATTACK MODEL' cell to generate them.")
else:
    # Load model
    model = joblib.load(MODEL_PATH)
    features = joblib.load(FEATURE_PATH)

    # Gemini
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    client = genai.Client(api_key=GEMINI_API_KEY)

    # ------------------------------------------------------------
    # CHANGE THIS FILE TO TEST
    # ------------------------------------------------------------
    file = "/content/attack_101.csv"

    if not os.path.exists(file):
        print(f"❌ Error: The dataset file '{file}' was not found. Please re-upload your data.")
    else:
        df = pd.read_csv(file)

        X = df[features].copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median())

        # Prediction
        prediction = model.predict(X)[0]

        # Probability
        probabilities = model.predict_proba(X)[0]
        confidence = max(probabilities) * 100

        # ------------------------------------------------------------
        # Collect network metrics
        # ------------------------------------------------------------

        metrics = {}

        for col in features:
            metrics[col] = float(X.iloc[0][col])

        # ------------------------------------------------------------
        # Gemini prompt
        # ------------------------------------------------------------

        prompt = f"""
        You are a Network-on-Chip security analyst.

        Analyze the following ML detection result.

        Predicted class:
        {prediction}

        ML confidence:
        {confidence:.2f}%

        Network metrics:
        {metrics}

        Important:
        - Do NOT invent an attack type that is not supported by the ML label.
        - "NETWORK ATTACK" means the model detected the attack scenario represented by that dataset.
        - "NODE-27 ATTACK" means the dataset represents an attack involving node 27.
        - Explain the evidence using the actual metrics.
        - Do not claim that the attack is specifically DDoS, DoS, flooding, etc. unless the provided label supports it.

        Return exactly these sections:

        1. Detection
        2. Confidence
        3. Key Indicators
        4. Why This Was Detected
        5. Security Impact
        6. Recommended Action

        Keep the explanation concise and suitable for a cybersecurity dashboard.
        """

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        # ------------------------------------------------------------
        # DISPLAY RESULT
        # ------------------------------------------------------------

        print("=" * 70)
        print("NETWORK SECURITY AI ANALYSIS")
        print("=" * 70)

        print(f"\nPrediction : {prediction}")
        print(f"Confidence : {confidence:.2f}%")

        print("\nAI EXPLANATION:")
        print(response.text)

        print("\n" + "=" * 70)


In [ ]:
# ============================================================
# STEP 3 — AUTO-RESTORE MODEL + GEMINI EXPLANATION
# ============================================================

import glob, os
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from google import genai
from google.colab import userdata

MODEL_PATH = "/content/multiclass_network_attack_detector.pkl"
FEATURE_PATH = "/content/multiclass_feature_names.pkl"

model_ready = False

# ------------------------------------------------------------
# 1. RESTORE / TRAIN MODEL
# ------------------------------------------------------------

if not os.path.exists(MODEL_PATH):

    print("Model not found — rebuilding automatically...")

    files = sorted(glob.glob("/content/*.csv"))
    frames = []

    if not files:
        print("❌ Error: No CSV files found. Please scroll up and re-upload your dataset.")
    else:
        for f in files:
            name = os.path.basename(f).lower()
            df = pd.read_csv(f)

            if name.startswith("normal"):
                label = "NORMAL"
            elif name == "attack_node27.csv":
                label = "NODE-27 ATTACK"
            elif name.startswith("attack_"):
                label = "NETWORK ATTACK"
            else:
                continue

            df["attack_type"] = label
            frames.append(df)

        if not frames:
            print("❌ Error: No valid labeled CSV files found.")
        else:
            data = pd.concat(frames, ignore_index=True)

            # Numeric features only
            features = [
                c for c in data.select_dtypes(include=np.number).columns
                if c not in ["rnd_generator_seed"]
            ]

            # Remove constant features
            features = [
                c for c in features
                if data[c].nunique() > 1
            ]

            X = data[features].copy()
            y = data["attack_type"]

            X = X.replace([np.inf, -np.inf], np.nan)
            X = X.fillna(X.median())

            model = RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1
            )

            model.fit(X, y)

            joblib.dump(model, MODEL_PATH)
            joblib.dump(features, FEATURE_PATH)

            print("MODEL REBUILT SUCCESSFULLY")
            model_ready = True

else:

    model = joblib.load(MODEL_PATH)
    features = joblib.load(FEATURE_PATH)

    print("MODEL LOADED SUCCESSFULLY")
    model_ready = True


# ------------------------------------------------------------
# 2. SELECT TEST FILE & PREDICT
# ------------------------------------------------------------

if model_ready:
    file = "/content/attack_101.csv"

    if not os.path.exists(file):
        print(f"❌ Error: The test file '{file}' was not found. Please upload it.")
    else:
        df = pd.read_csv(file)

        X_test = df[features].copy()
        X_test = X_test.replace([np.inf, -np.inf], np.nan)
        X_test = X_test.fillna(X_test.median())

        # ------------------------------------------------------------
        # 3. PREDICTION
        # ------------------------------------------------------------

        prediction = model.predict(X_test)[0]
        probabilities = model.predict_proba(X_test)[0]
        confidence = max(probabilities) * 100

        print("\n" + "="*65)
        print("NETWORK SECURITY DETECTION")
        print("="*65)

        print("File       :", os.path.basename(file))
        print("Prediction :", prediction)
        print(f"Confidence : {confidence:.2f}%")

        print("\nClass probabilities:")

        for cls, prob in zip(model.classes_, probabilities):
            print(f"{cls}: {prob*100:.2f}%")

        # ------------------------------------------------------------
        # 4. GEMINI EXPLANATION
        # ------------------------------------------------------------
        try:
            GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
            client = genai.Client(api_key=GEMINI_API_KEY)

            metrics = {
                col: float(X_test.iloc[0][col])
                for col in features
            }

            prompt = f"""
            You are an expert Network-on-Chip cybersecurity analyst.

            Analyze this ML detection result.

            Predicted class:
            {prediction}

            Confidence:
            {confidence:.2f}%

            Network metrics:
            {metrics}

            Important rules:
            - Do not invent a specific attack name.
            - NETWORK ATTACK means the trained dataset's attack scenario.
            - NODE-27 ATTACK means the trained dataset's node-27 scenario.
            - Explain only what the metrics support.
            - Do not call it DDoS, DoS, flooding, etc. unless supported by the label.

            Give a concise dashboard explanation with exactly:

            DETECTION:
            CONFIDENCE:
            KEY INDICATORS:
            WHY DETECTED:
            SECURITY IMPACT:
            RECOMMENDED ACTION:
            """

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            print("\n" + "="*65)
            print("GEMINI SECURITY EXPLANATION")
            print("="*65)
            print(response.text)
            print("\nDONE.")
        except Exception as e:
            print(f"\n❌ Gemini API Error: {e}")


In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded:", len(uploaded), "files")

In [ ]:
import glob, os
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier

files = sorted(glob.glob("/content/*.csv"))

print("CSV files found:", len(files))

frames = []

for f in files:
    name = os.path.basename(f).lower()
    df = pd.read_csv(f)

    if name.startswith("normal"):
        label = "NORMAL"
    elif name == "attack_node27.csv":
        label = "NODE-27 ATTACK"
    elif name.startswith("attack_"):
        label = "NETWORK ATTACK"
    else:
        continue

    df["attack_type"] = label
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

features = [
    c for c in data.select_dtypes(include=np.number).columns
    if c != "rnd_generator_seed"
]

features = [
    c for c in features
    if data[c].nunique() > 1
]

X = data[features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

y = data["attack_type"]

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X, y)

joblib.dump(model, "/content/multiclass_network_attack_detector.pkl")
joblib.dump(features, "/content/multiclass_feature_names.pkl")

print("\n================================")
print("MODEL READY")
print("================================")
print("Samples:", len(data))
print("Classes:", list(model.classes_))
print("Features:", features)

In [ ]:
for file in [
    "/content/normal_1.csv",
    "/content/attack_101.csv",
    "/content/attack_node27.csv"
]:

    df = pd.read_csv(file)
    X = df[features]

    prediction = model.predict(X)[0]
    probs = model.predict_proba(X)[0]

    print("\n", os.path.basename(file))
    print("Prediction:", prediction)
    print("Confidence:", f"{max(probs)*100:.2f}%")

    for cls, p in zip(model.classes_, probs):
        print(f"  {cls}: {p*100:.2f}%")

In [ ]:
import os
import glob

print("=== FILES AVAILABLE IN COLAB ===")

for f in sorted(glob.glob("/content/*")):
    print(os.path.basename(f))

print("\n=== ATTACK-RELATED FILES ===")

matches = []

for root, dirs, files in os.walk("/content"):
    for name in files:
        if any(x in name.lower() for x in [
            "attack", "node27", "noxim", "config"
        ]):
            path = os.path.join(root, name)
            matches.append(path)

if matches:
    for x in matches:
        print(x)
else:
    print("NO ATTACK SOURCE/CONFIG FILES FOUND")

In [ ]:
# ============================================================
# FINAL: EXPLAINABLE DETECTION + GEMINI
# ============================================================

from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. FULL DATASET — IMPORTANT: keep all 25 samples
# ------------------------------------------------------------

X_full = data[features].copy()
y_full = data["attack_type"].copy()

X_full = X_full.replace([np.inf, -np.inf], np.nan)
X_full = X_full.fillna(X_full.median())

# ------------------------------------------------------------
# 2. FILE WE WANT TO ANALYZE
# ------------------------------------------------------------

file = "/content/attack_101.csv"

test_df = pd.read_csv(file)

X_test = test_df[features].copy()
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Use training medians
X_test = X_test.fillna(X_full.median())

# ------------------------------------------------------------
# 3. PREDICTION
# ------------------------------------------------------------

prediction = model.predict(X_test)[0]

probabilities = model.predict_proba(X_test)[0]

confidence = max(probabilities) * 100

# ------------------------------------------------------------
# 4. MODEL EXPLAINABILITY
# ------------------------------------------------------------

result = permutation_importance(
    model,
    X_full,
    y_full,
    scoring="accuracy",
    n_repeats=10,
    random_state=42
)

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": result.importances_mean
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

top_indicators = importance_df.head(5)

# ------------------------------------------------------------
# 5. DISPLAY DETECTION
# ------------------------------------------------------------

print("=" * 70)
print("NETWORK SECURITY AI")
print("=" * 70)

print("File:", file.split("/")[-1])
print("Detection:", prediction)
print(f"Confidence: {confidence:.2f}%")

print("\nCLASS PROBABILITIES:")

for cls, prob in zip(model.classes_, probabilities):
    print(f"{cls}: {prob*100:.2f}%")

print("\nTOP EXPLAINABLE INDICATORS:")
print(top_indicators.to_string(index=False))

# ------------------------------------------------------------
# 6. ACTUAL NETWORK VALUES
# ------------------------------------------------------------

metrics = {
    feature: float(X_test.iloc[0][feature])
    for feature in features
}

# ------------------------------------------------------------
# 7. GEMINI
# ------------------------------------------------------------

prompt = f"""
You are a Network-on-Chip cybersecurity analyst.

Analyze this machine-learning detection.

DETECTION:
{prediction}

CONFIDENCE:
{confidence:.2f}%

TOP MODEL INDICATORS:
{top_indicators.to_dict("records")}

ACTUAL NETWORK METRICS:
{metrics}

Explain the detection using only the supplied evidence.

IMPORTANT:
- Do not invent an attack type.
- NETWORK ATTACK is the available attack class.
- NODE-27 ATTACK is the available node-27 scenario.
- Do not call it DDoS, DoS, flooding, etc. unless supported by the data.
- Explain why the traffic appears normal or anomalous.
- Mention the relevant network-performance indicators.

Return exactly:

ATTACK STATUS:
ATTACK TYPE:
CONFIDENCE:
WHY DETECTED:
KEY INDICATORS:
POTENTIAL IMPACT:
RECOMMENDED ACTION:
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("\n" + "=" * 70)
print("GEMINI EXPLAINABILITY")
print("=" * 70)

print(response.text)

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("=" * 70)
print("GEMINI EXPLAINABILITY")
print("=" * 70)
print(response.text)

In [ ]:
# ============================================================
# NoC SECURITY THREAT LIBRARY — COLAB
# ============================================================

from IPython.display import display, HTML

threats = [
    {
        "name": "On-Chip Denial of Service (DoS)",
        "severity": "HIGH",
        "category": "Availability",
        "description": "Excessive traffic degrades NoC availability and performance.",
        "indicators": "↑ latency • ↑ congestion • ↓ throughput",
        "status": "KNOWLEDGE BASE"
    },
    {
        "name": "Traffic Flooding / Injection",
        "severity": "HIGH",
        "category": "Availability",
        "description": "Excessive packet or flit injection consumes NoC resources.",
        "indicators": "↑ packet volume • ↑ flits • ↑ delay",
        "status": "KNOWLEDGE BASE"
    },
    {
        "name": "Malicious / Compromised Node",
        "severity": "HIGH",
        "category": "Node Security",
        "description": "A compromised node generates abnormal or disruptive traffic.",
        "indicators": "abnormal traffic • latency anomalies • local congestion",
        "status": "MODEL SUPPORTED"
    },
    {
        "name": "Routing Manipulation",
        "severity": "HIGH",
        "category": "Integrity",
        "description": "Manipulated routing causes abnormal packet paths or congestion.",
        "indicators": "unexpected paths • latency anomalies • congestion",
        "status": "KNOWLEDGE BASE"
    },
    {
        "name": "Hardware Trojan",
        "severity": "CRITICAL",
        "category": "Hardware Security",
        "description": "Malicious hardware logic alters NoC behaviour.",
        "indicators": "power anomalies • traffic anomalies • performance changes",
        "status": "KNOWLEDGE BASE"
    },
    {
        "name": "NoC Side-Channel Attack",
        "severity": "MEDIUM",
        "category": "Confidentiality",
        "description": "Timing, contention or power behaviour is exploited to infer information.",
        "indicators": "timing variations • contention • latency correlation",
        "status": "KNOWLEDGE BASE"
    }
]

cards = ""

for t in threats:

    badge = (
        "<span class='supported'>✓ MODEL SUPPORTED</span>"
        if t["status"] == "MODEL SUPPORTED"
        else "<span class='knowledge'>KNOWLEDGE BASE</span>"
    )

    cards += f"""
    <div class="card">
        <div class="top">
            <span class="severity">{t['severity']}</span>
            {badge}
        </div>

        <h2>{t['name']}</h2>

        <div class="category">{t['category']}</div>

        <p>{t['description']}</p>

        <div class="label">KEY INDICATORS</div>
        <div class="indicators">{t['indicators']}</div>
    </div>
    """

html = f"""
<style>

body {{
    font-family: Arial, sans-serif;
}}

.title {{
    font-size: 30px;
    font-weight: 700;
    margin-bottom: 8px;
}}

.subtitle {{
    color: #666;
    margin-bottom: 25px;
}}

.grid {{
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 18px;
}}

.card {{
    border: 1px solid #ddd;
    border-radius: 16px;
    padding: 22px;
    background: white;
    box-shadow: 0 3px 12px rgba(0,0,0,0.06);
}}

.top {{
    display: flex;
    justify-content: space-between;
    align-items: center;
}}

.severity {{
    font-size: 12px;
    font-weight: bold;
    padding: 5px 9px;
    border-radius: 8px;
    background: #ffe5e5;
    color: #b00020;
}}

.supported {{
    font-size: 11px;
    font-weight: bold;
    color: #087f23;
    background: #e5f7e9;
    padding: 5px 8px;
    border-radius: 8px;
}}

.knowledge {{
    font-size: 11px;
    font-weight: bold;
    color: #555;
    background: #eeeeee;
    padding: 5px 8px;
    border-radius: 8px;
}}

h2 {{
    margin-bottom: 5px;
}}

.category {{
    color: #777;
    font-size: 13px;
    margin-bottom: 12px;
}}

p {{
    line-height: 1.5;
}}

.label {{
    font-size: 11px;
    font-weight: bold;
    color: #777;
    margin-top: 18px;
}}

.indicators {{
    margin-top: 6px;
    font-size: 13px;
}}

</style>

<div class="title">NoC Security Threat Library</div>

<div class="subtitle">
Known Network-on-Chip security threats and their observable indicators.
</div>

<div class="grid">
{cards}
</div>

<p style="margin-top:25px;color:#777;font-size:12px;">
⚠️ Threat Library entries marked "KNOWLEDGE BASE" are security knowledge,
not predictions from the current ML model.
</p>
"""

display(HTML(html))

In [ ]:
# ============================================================
# FINAL NOC SECURITY AI DEMO
# ML + EXPLAINABILITY + GEMINI + THREAT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display, HTML
from sklearn.inspection import permutation_importance

# ------------------------------------------------------------
# SELECT FILE
# ------------------------------------------------------------

FILE = "/content/attack_101.csv"

# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

test_df = pd.read_csv(FILE)

X_test = test_df[features].copy()
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(X_full.median())

prediction = model.predict(X_test)[0]
probabilities = model.predict_proba(X_test)[0]
confidence = max(probabilities) * 100

# ------------------------------------------------------------
# EXPLAINABILITY
# ------------------------------------------------------------

perm = permutation_importance(
    model,
    X_full,
    y_full,
    scoring="accuracy",
    n_repeats=10,
    random_state=42
)

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

top5 = importance_df.head(5)

# ------------------------------------------------------------
# GEMINI
# ------------------------------------------------------------

metrics = {
    f: float(X_test.iloc[0][f])
    for f in features
}

prompt = f"""
You are a Network-on-Chip cybersecurity analyst.

ML RESULT:
Prediction: {prediction}
Confidence: {confidence:.2f}%

TOP MODEL INDICATORS:
{top5.to_dict("records")}

NETWORK METRICS:
{metrics}

Explain the result without inventing an attack type.

Important:
NETWORK ATTACK and NODE-27 ATTACK are dataset scenarios.
Do not call them DDoS, DoS, flooding, Trojan, etc.

Return exactly:

WHY DETECTED:
SECURITY IMPACT:
RECOMMENDED ACTION:
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# ------------------------------------------------------------
# THREAT LIBRARY
# ------------------------------------------------------------

threats = [
    ("On-Chip Denial of Service", "HIGH", "Availability"),
    ("Traffic Flooding / Injection", "HIGH", "Availability"),
    ("Malicious / Compromised Node", "HIGH", "Node Security"),
    ("Routing Manipulation", "HIGH", "Integrity"),
    ("Hardware Trojan", "CRITICAL", "Hardware Security"),
    ("NoC Side-Channel Attack", "MEDIUM", "Confidentiality")
]

# ------------------------------------------------------------
# DASHBOARD
# ------------------------------------------------------------

prob_html = ""

for cls, prob in zip(model.classes_, probabilities):
    prob_html += f"""
    <div class="prob">
        <span>{cls}</span>
        <b>{prob*100:.2f}%</b>
    </div>
    """

indicator_html = ""

for _, row in top5.iterrows():
    indicator_html += f"""
    <div class="indicator">
        <span>{row['Feature']}</span>
        <b>{row['Importance']:.4f}</b>
    </div>
    """

threat_html = ""

for name, severity, category in threats:

    supported = (
        name == "Malicious / Compromised Node"
    )

    badge = (
        "✓ MODEL SUPPORTED"
        if supported
        else "KNOWLEDGE BASE"
    )

    threat_html += f"""
    <div class="threat">
        <div>
            <b>{name}</b>
            <small>{category}</small>
        </div>

        <span class="badge">{severity}</span>
        <span class="status">{badge}</span>
    </div>
    """

dashboard = f"""
<style>

.container {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
}}

.header {{
    padding: 25px;
    border-radius: 18px;
    background: #111;
    color: white;
}}

.header h1 {{
    margin: 0;
}}

.grid {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 18px;
    margin-top: 18px;
}}

.card {{
    background: white;
    border: 1px solid #ddd;
    border-radius: 16px;
    padding: 22px;
}}

.detection {{
    font-size: 27px;
    font-weight: bold;
}}

.confidence {{
    font-size: 18px;
    margin-top: 8px;
}}

.prob, .indicator {{
    display: flex;
    justify-content: space-between;
    padding: 9px 0;
    border-bottom: 1px solid #eee;
}}

.indicator {{
    font-size: 13px;
}}

.threat {{
    display: grid;
    grid-template-columns: 1fr auto auto;
    gap: 10px;
    align-items: center;
    padding: 13px 0;
    border-bottom: 1px solid #eee;
}}

.threat small {{
    display: block;
    color: #777;
    margin-top: 3px;
}}

.badge {{
    background: #ffe5e5;
    color: #a00000;
    padding: 5px 8px;
    border-radius: 7px;
    font-size: 10px;
    font-weight: bold;
}}

.status {{
    background: #eeeeee;
    padding: 5px 8px;
    border-radius: 7px;
    font-size: 10px;
}}

.explanation {{
    line-height: 1.6;
    white-space: pre-wrap;
}}

.full {{
    grid-column: 1 / -1;
}}

</style>

<div class="container">

<div class="header">
    <h1>Network-on-Chip Security AI</h1>
    <p>Explainable ML-based network attack analysis</p>
</div>

<div class="grid">

<div class="card">
<h2>Detection</h2>

<div class="detection">{prediction}</div>

<div class="confidence">
Confidence: {confidence:.2f}%
</div>

<h3>Class Probabilities</h3>

{prob_html}

</div>


<div class="card">

<h2>Explainable Indicators</h2>

{indicator_html}

<p style="color:#777;font-size:12px">
Indicators show which features the trained model relies on.
They are evidence for the model decision, not proof of an attack by themselves.
</p>

</div>


<div class="card full">

<h2>Gemini Security Analysis</h2>

<div class="explanation">
{response.text}
</div>

</div>


<div class="card full">

<h2>Threat Library</h2>

<p style="color:#777">
Known Network-on-Chip security threats.
Knowledge-base entries are NOT ML predictions.
</p>

{threat_html}

</div>

</div>

</div>
"""

display(HTML(dashboard))

In [ ]:
# ============================================================
# FINAL INTERACTIVE NoC SECURITY WEB APP
# ============================================================

!pip -q install -U gradio

import gradio as gr
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance

# ------------------------------------------------------------
# THREAT LIBRARY
# ------------------------------------------------------------

THREATS = [
    ("On-Chip Denial of Service", "HIGH", "Availability"),
    ("Traffic Flooding / Injection", "HIGH", "Availability"),
    ("Malicious / Compromised Node", "HIGH", "Node Security"),
    ("Routing Manipulation", "HIGH", "Integrity"),
    ("Hardware Trojan", "CRITICAL", "Hardware Security"),
    ("NoC Side-Channel Attack", "MEDIUM", "Confidentiality")
]

# ------------------------------------------------------------
# ANALYSIS FUNCTION
# ------------------------------------------------------------

def analyze_csv(file):

    if file is None:
        return "Please upload a CSV file.", "", "", ""

    try:
        path = file

        df = pd.read_csv(path)

        # Validate required features
        missing = [c for c in features if c not in df.columns]

        if missing:
            return (
                "INVALID CSV",
                f"Missing features: {missing}",
                "",
                ""
            )

        # Prepare input
        sample = df[features].copy()
        sample = sample.replace([np.inf, -np.inf], np.nan)
        sample = sample.fillna(X_full.median())

        # Prediction
        prediction = model.predict(sample)[0]
        probabilities = model.predict_proba(sample)[0]
        confidence = max(probabilities) * 100

        # ----------------------------------------------------
        # CLASS PROBABILITIES
        # ----------------------------------------------------

        probability_html = ""

        for cls, p in zip(model.classes_, probabilities):

            probability_html += f"""
            <div class="prob-row">
                <span>{cls}</span>
                <b>{p*100:.2f}%</b>
            </div>
            """

        # ----------------------------------------------------
        # EXPLAINABILITY
        # ----------------------------------------------------

        perm = permutation_importance(
            model,
            X_full,
            y_full,
            scoring="accuracy",
            n_repeats=10,
            random_state=42
        )

        importance_df = pd.DataFrame({
            "Feature": features,
            "Importance": perm.importances_mean
        }).sort_values(
            "Importance",
            ascending=False
        )

        top5 = importance_df.head(5)

        indicator_html = ""

        for _, row in top5.iterrows():

            indicator_html += f"""
            <div class="indicator-row">
                <span>{row['Feature']}</span>
                <b>{row['Importance']:.4f}</b>
            </div>
            """

        # ----------------------------------------------------
        # GEMINI
        # ----------------------------------------------------

        metrics = {
            f: float(sample.iloc[0][f])
            for f in features
        }

        prompt = f"""
You are a Network-on-Chip cybersecurity analyst.

ML DETECTION:
{prediction}

CONFIDENCE:
{confidence:.2f}%

TOP MODEL INDICATORS:
{top5.to_dict("records")}

NETWORK METRICS:
{metrics}

Explain this result.

Rules:
- Do not invent an attack type.
- NETWORK ATTACK and NODE-27 ATTACK are the available ML scenarios.
- Do not call it DDoS, DoS, flooding, Trojan, etc. unless supported.
- Explain the evidence clearly.

Return:

WHY DETECTED:
SECURITY IMPACT:
RECOMMENDED ACTION:
"""

        try:

            gemini_response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            explanation = gemini_response.text

        except Exception as e:

            explanation = f"Gemini unavailable: {str(e)}"

        # ----------------------------------------------------
        # THREAT LIBRARY
        # ----------------------------------------------------

        threat_html = ""

        for name, severity, category in THREATS:

            supported = (
                name == "Malicious / Compromised Node"
            )

            status = (
                "✓ MODEL SUPPORTED"
                if supported
                else "KNOWLEDGE BASE"
            )

            threat_html += f"""
            <div class="threat-row">

                <div>
                    <b>{name}</b>
                    <small>{category}</small>
                </div>

                <span class="severity">
                    {severity}
                </span>

                <span class="status">
                    {status}
                </span>

            </div>
            """

        # ----------------------------------------------------
        # DETECTION CARD
        # ----------------------------------------------------

        detection = f"""
        <div class="detection-card">

            <div class="small-label">
                AI SECURITY DETECTION
            </div>

            <div class="prediction">
                {prediction}
            </div>

            <div class="confidence">
                Confidence: <b>{confidence:.2f}%</b>
            </div>

        </div>
        """

        return (
            detection,
            probability_html,
            indicator_html,
            explanation,
            threat_html
        )

    except Exception as e:

        return (
            "ERROR",
            str(e),
            "",
            "",
            ""
        )


# ------------------------------------------------------------
# CSS
# ------------------------------------------------------------

CSS = """

body {
    background: #f5f7fa;
}

.gradio-container {
    max-width: 1250px !important;
}

.title {
    font-size: 34px;
    font-weight: 800;
}

.subtitle {
    color: #6b7280;
    margin-bottom: 25px;
}

.detection-card {
    background: #111827;
    color: white;
    padding: 28px;
    border-radius: 18px;
}

.small-label {
    font-size: 12px;
    color: #9ca3af;
    letter-spacing: 1px;
}

.prediction {
    font-size: 32px;
    font-weight: 800;
    margin-top: 10px;
}

.confidence {
    margin-top: 10px;
    font-size: 18px;
}

.prob-row,
.indicator-row {
    display: flex;
    justify-content: space-between;
    padding: 12px;
    border-bottom: 1px solid #e5e7eb;
}

.threat-row {
    display: grid;
    grid-template-columns: 1fr auto auto;
    gap: 12px;
    padding: 14px;
    border-bottom: 1px solid #e5e7eb;
}

.threat-row small {
    display: block;
    color: #777;
    margin-top: 3px;
}

.severity {
    background: #fee2e2;
    color: #991b1b;
    padding: 5px 9px;
    border-radius: 7px;
    font-size: 11px;
    font-weight: 700;
}

.status {
    background: #f3f4f6;
    padding: 5px 9px;
    border-radius: 7px;
    font-size: 11px;
}

.explanation {
    line-height: 1.7;
    white-space: pre-wrap;
}
"""


# ------------------------------------------------------------
# GRADIO APP
# ------------------------------------------------------------

with gr.Blocks(
    css=CSS,
    title="NoC Security AI"
) as demo:

    gr.Markdown("""
    # 🛡️ Network-on-Chip Security AI

    **Explainable ML-based Network Attack Detection**

    Upload a Noxim network simulation CSV and receive
    an AI-powered security analysis.
    """)

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("### 📂 Upload Network Data")

            upload = gr.File(
                label="Upload Noxim CSV",
                file_types=[".csv"],
                type="filepath"
            )

            analyze = gr.Button(
                "🔍 ANALYZE NETWORK",
                variant="primary"
            )

            gr.Markdown("""
            **Pipeline**

            CSV → ML Detection → Confidence
            → Explainable Indicators → Gemini Analysis
            """)

        with gr.Column(scale=2):

            detection_output = gr.HTML(
                label="Detection"
            )

    gr.Markdown("## 📊 Classification Confidence")

    probabilities_output = gr.HTML()

    gr.Markdown("## 🔎 Explainable Indicators")

    indicators_output = gr.HTML()

    gr.Markdown("## 🤖 Gemini Security Analysis")

    explanation_output = gr.Markdown(
        elem_classes=["explanation"]
    )

    gr.Markdown("## 🛡️ NoC Threat Library")

    gr.Markdown(
        "Known security threats. "
        "**Knowledge Base entries are not ML predictions.**"
    )

    threats_output = gr.HTML()

    analyze.click(
        fn=analyze_csv,
        inputs=upload,
        outputs=[
            detection_output,
            probabilities_output,
            indicators_output,
            explanation_output,
            threats_output
        ]
    )

# ------------------------------------------------------------
# LAUNCH
# ------------------------------------------------------------

demo.launch()

In [ ]:
# ============================================================
# NoC SENTINEL — POLISHED CYBERSECURITY DASHBOARD
# ============================================================

import gradio as gr

# Close previous Gradio instance if it exists
try:
    demo.close()
except:
    pass


CSS = """
/* ---------- GLOBAL ---------- */

.gradio-container {
    max-width: 1350px !important;
    margin: auto !important;
    background:
        radial-gradient(circle at 10% 0%, rgba(37,99,235,.08), transparent 30%),
        #f7f9fc !important;
    font-family: Inter, Arial, sans-serif !important;
}

footer {
    display: none !important;
}

/* ---------- HEADER ---------- */

.hero {
    background: linear-gradient(135deg, #0b1220, #111c33);
    color: white;
    border-radius: 24px;
    padding: 34px 38px;
    margin-bottom: 22px;
    box-shadow: 0 15px 40px rgba(15,23,42,.16);
}

.hero-title {
    font-size: 34px;
    font-weight: 800;
    letter-spacing: -1px;
}

.hero-subtitle {
    color: #aab6ca;
    font-size: 15px;
    margin-top: 8px;
}

.hero-pill {
    display: inline-block;
    margin-top: 18px;
    padding: 7px 12px;
    border-radius: 999px;
    background: rgba(59,130,246,.14);
    border: 1px solid rgba(96,165,250,.25);
    color: #93c5fd;
    font-size: 12px;
    font-weight: 600;
}

/* ---------- PANELS ---------- */

.panel {
    background: white;
    border: 1px solid #e7ebf2;
    border-radius: 20px;
    padding: 22px;
    box-shadow: 0 8px 25px rgba(15,23,42,.05);
}

.section-title {
    font-size: 20px;
    font-weight: 750;
    color: #111827;
    margin-bottom: 5px;
}

.section-description {
    color: #6b7280;
    font-size: 13px;
    margin-bottom: 18px;
}

/* ---------- UPLOAD ---------- */

.upload-panel {
    min-height: 250px;
}

.analyze-btn {
    border-radius: 12px !important;
    font-weight: 700 !important;
    min-height: 48px !important;
}

/* ---------- DETECTION ---------- */

.detection-card {
    min-height: 190px;
    border-radius: 20px;
    padding: 28px;
    background: linear-gradient(135deg, #0f172a, #172554);
    color: white;
    box-shadow: 0 12px 30px rgba(15,23,42,.18);
}

.detection-label {
    color: #93c5fd;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1.5px;
}

.detection-value {
    font-size: 34px;
    font-weight: 850;
    margin-top: 12px;
    letter-spacing: -1px;
}

.detection-confidence {
    margin-top: 12px;
    font-size: 16px;
    color: #cbd5e1;
}

.detection-confidence strong {
    color: white;
}

/* ---------- PROBABILITIES ---------- */

.prob-row {
    margin: 13px 0;
}

.prob-header {
    display: flex;
    justify-content: space-between;
    font-size: 13px;
    font-weight: 650;
    color: #374151;
    margin-bottom: 6px;
}

.prob-track {
    height: 8px;
    background: #edf0f5;
    border-radius: 99px;
    overflow: hidden;
}

.prob-fill {
    height: 100%;
    border-radius: 99px;
    background: linear-gradient(90deg, #2563eb, #60a5fa);
}

/* ---------- INDICATORS ---------- */

.indicator-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 14px 0;
    border-bottom: 1px solid #eef1f5;
}

.indicator-name {
    color: #374151;
    font-family: monospace;
    font-size: 12px;
}

.indicator-value {
    color: #2563eb;
    font-weight: 750;
    font-size: 13px;
}

/* ---------- GEMINI ---------- */

.gemini-box {
    background: linear-gradient(135deg, #f8fafc, #eef4ff);
    border: 1px solid #dbe7ff;
    border-radius: 18px;
    padding: 24px;
    line-height: 1.7;
    color: #1f2937;
}

.ai-badge {
    display: inline-block;
    background: #e0ecff;
    color: #1d4ed8;
    padding: 5px 9px;
    border-radius: 7px;
    font-size: 10px;
    font-weight: 800;
    margin-bottom: 12px;
}

/* ---------- THREAT LIBRARY ---------- */

.threat-row {
    display: grid;
    grid-template-columns: 1fr auto auto;
    align-items: center;
    gap: 15px;
    padding: 16px 4px;
    border-bottom: 1px solid #edf0f4;
}

.threat-name {
    font-weight: 700;
    color: #111827;
}

.threat-category {
    color: #8a94a6;
    font-size: 11px;
    margin-top: 3px;
}

.severity {
    font-size: 10px;
    font-weight: 800;
    padding: 6px 9px;
    border-radius: 7px;
    background: #fff1f2;
    color: #be123c;
}

.status-supported {
    font-size: 10px;
    font-weight: 750;
    padding: 6px 9px;
    border-radius: 7px;
    background: #ecfdf5;
    color: #047857;
}

.status-library {
    font-size: 10px;
    font-weight: 700;
    padding: 6px 9px;
    border-radius: 7px;
    background: #f3f4f6;
    color: #6b7280;
}

/* ---------- PIPELINE ---------- */

.pipeline {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 18px;
    background: #f8fafc;
    border-radius: 15px;
    border: 1px solid #e8edf4;
    margin-top: 18px;
}

.pipeline-step {
    text-align: center;
    font-size: 11px;
    color: #475569;
    font-weight: 650;
}

.pipeline-icon {
    font-size: 20px;
    margin-bottom: 5px;
}

.arrow {
    color: #94a3b8;
    font-size: 18px;
}

/* ---------- MOBILE ---------- */

@media(max-width: 800px) {
    .hero-title {
        font-size: 26px;
    }

    .pipeline {
        flex-wrap: wrap;
        gap: 15px;
    }

    .threat-row {
        grid-template-columns: 1fr;
    }
}
"""


# ============================================================
# BUILD HTML HELPERS
# ============================================================

def make_probabilities(html):
    # Existing probability HTML is converted into cleaner bars
    return html


def make_threats():
    rows = ""

    for name, severity, category in THREATS:

        if name == "Malicious / Compromised Node":
            status = """
            <span class="status-supported">
                ✓ MODEL SUPPORTED
            </span>
            """
        else:
            status = """
            <span class="status-library">
                KNOWLEDGE BASE
            </span>
            """

        rows += f"""
        <div class="threat-row">

            <div>
                <div class="threat-name">{name}</div>
                <div class="threat-category">{category}</div>
            </div>

            <span class="severity">{severity}</span>

            {status}

        </div>
        """

    return rows


# ============================================================
# WRAPPER AROUND YOUR EXISTING ANALYSIS FUNCTION
# ============================================================

def polished_analyze(file):

    result = analyze_csv(file)

    detection = result[0]
    probabilities = result[1]
    indicators = result[2]
    explanation = result[3]

    # --------------------------------------------------------
    # Detection card
    # --------------------------------------------------------

    detection_clean = detection

    import re

    prediction_match = re.search(
        r'<div class="prediction">(.*?)</div>',
        detection_clean,
        re.S
    )

    confidence_match = re.search(
        r'Confidence:\s*<b>(.*?)%</b>',
        detection_clean,
        re.S
    )

    prediction = (
        prediction_match.group(1).strip()
        if prediction_match
        else "ANALYSIS COMPLETE"
    )

    confidence = (
        confidence_match.group(1).strip()
        if confidence_match
        else "—"
    )

    detection_html = f"""
    <div class="detection-card">

        <div class="detection-label">
            AI SECURITY DETECTION
        </div>

        <div class="detection-value">
            {prediction}
        </div>

        <div class="detection-confidence">
            Confidence:
            <strong>{confidence}%</strong>
        </div>

    </div>
    """

    # --------------------------------------------------------
    # Threat library
    # --------------------------------------------------------

    threat_html = f"""
    <div class="panel">

        <div class="section-title">
            NoC Threat Library
        </div>

        <div class="section-description">
            Security knowledge base. These entries are not automatically
            classified by the current ML model.
        </div>

        {make_threats()}

    </div>
    """

    # --------------------------------------------------------
    # Gemini
    # --------------------------------------------------------

    gemini_html = f"""
    <div class="gemini-box">

        <div class="ai-badge">
            GEMINI SECURITY ANALYST
        </div>

        <div class="section-title">
            AI Security Analysis
        </div>

        <div style="margin-top:15px;">
            {explanation}
        </div>

    </div>
    """

    return (
        detection_html,
        probabilities,
        indicators,
        gemini_html,
        threat_html
    )


# ============================================================
# APP
# ============================================================

with gr.Blocks() as demo:

    # --------------------------------------------------------
    # HERO
    # --------------------------------------------------------

    gr.HTML("""
    <div class="hero">

        <div class="hero-title">
            🛡️ Network-on-Chip Security AI
        </div>

        <div class="hero-subtitle">
            Explainable machine-learning security analysis
            for Noxim Network-on-Chip simulations.
        </div>

        <div class="hero-pill">
            ML DETECTION  •  EXPLAINABILITY  •  GEMINI AI
        </div>

    </div>
    """)

    # --------------------------------------------------------
    # PIPELINE
    # --------------------------------------------------------

    gr.HTML("""
    <div class="pipeline">

        <div class="pipeline-step">
            <div class="pipeline-icon">📄</div>
            Noxim CSV
        </div>

        <div class="arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">🧠</div>
            ML Detection
        </div>

        <div class="arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">📊</div>
            Confidence
        </div>

        <div class="arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">🔎</div>
            Explainability
        </div>

        <div class="arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">✨</div>
            Gemini Analysis
        </div>

    </div>
    """)

    # --------------------------------------------------------
    # INPUT + DETECTION
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column(scale=1):

            gr.HTML("""
            <div class="panel upload-panel">

                <div class="section-title">
                    📂 Analyze Network Simulation
                </div>

                <div class="section-description">
                    Upload a Noxim CSV generated from your
                    Network-on-Chip simulation.
                </div>

            </div>
            """)

            upload = gr.File(
                label="Noxim Simulation CSV",
                file_types=[".csv"],
                type="filepath"
            )

            analyze_btn = gr.Button(
                "🔍  ANALYZE NETWORK",
                variant="primary",
                elem_classes="analyze-btn"
            )

        with gr.Column(scale=1):

            detection_output = gr.HTML()

    # --------------------------------------------------------
    # CONFIDENCE + INDICATORS
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column():

            gr.HTML("""
            <div class="section-title">
                📊 Classification Confidence
            </div>

            <div class="section-description">
                Probability assigned by the trained classifier.
            </div>
            """)

            probability_output = gr.HTML()

        with gr.Column():

            gr.HTML("""
            <div class="section-title">
                🔎 Model Feature Importance
            </div>

            <div class="section-description">
                Features that most influence the trained model.
            </div>
            """)

            indicator_output = gr.HTML()

    # --------------------------------------------------------
    # GEMINI
    # --------------------------------------------------------

    gr.HTML("""
    <div style="margin-top:25px;">
        <div class="section-title">
            ✨ Gemini Security Analysis
        </div>

        <div class="section-description">
            AI-generated interpretation of the ML result and
            recommended security response.
        </div>
    </div>
    """)

    gemini_output = gr.HTML()

    # --------------------------------------------------------
    # THREAT LIBRARY
    # --------------------------------------------------------

    gr.HTML("""
    <div style="margin-top:25px;">
        <div class="section-title">
            🛡️ NoC Threat Library
        </div>

        <div class="section-description">
            Known Network-on-Chip security threats and attack concepts.
        </div>
    </div>
    """)

    threats_output = gr.HTML()

    # --------------------------------------------------------
    # EVENT
    # --------------------------------------------------------

    analyze_btn.click(
        fn=polished_analyze,
        inputs=upload,
        outputs=[
            detection_output,
            probability_output,
            indicator_output,
            gemini_output,
            threats_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    theme=gr.themes.Soft(),
    css=CSS
)

In [ ]:
import glob, os

files = sorted(glob.glob("/content/*.csv"))

print("CSV COUNT:", len(files))
print("\n".join(os.path.basename(f) for f in files))

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", len(uploaded))

In [ ]:
import glob, os

csvs = glob.glob("/content/*.csv")

print("CSV COUNT:", len(csvs))
print([os.path.basename(x) for x in csvs])

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import os

rows = []

for name in uploaded:
    if name.startswith("NORMAL_"):
        label = "NORMAL"
    elif name.startswith("DOS_FLOODING_"):
        label = "DOS_FLOODING"
    elif name.startswith("BLACKHOLE_"):
        label = "BLACKHOLE"
    elif name.startswith("ROUTING_MANIPULATION_"):
        label = "ROUTING_MANIPULATION"
    else:
        continue

    df = pd.read_csv(name)
    df["label"] = label
    rows.append(df)

data = pd.concat(rows, ignore_index=True)

print("DATASET:", data.shape)
print(data["label"].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# FEATURES
# -----------------------------
features = [
    "total_received_packets",
    "total_received_flits",
    "received_ideal_flits_ratio",
    "global_average_delay_cycles",
    "max_delay_cycles",
    "network_throughput_flits_per_cycle",
    "average_ip_throughput_flits_per_cycle_per_ip",
    "total_energy_j",
    "dynamic_energy_j",
    "static_energy_j"
]

X = data[features].copy()
y = data["label"].copy()

# Remove invalid values
X = X.replace([np.inf, -np.inf], np.nan)
valid = X.notna().all(axis=1)

X = X.loc[valid]
y = y.loc[valid]

print("Samples:", len(X))
print("Features:", len(features))
print("\nClasses:")
print(y.value_counts())

# -----------------------------
# TRAIN / TEST
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# -----------------------------
# RANDOM FOREST
# -----------------------------
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# -----------------------------
# EVALUATION
# -----------------------------
pred = model.predict(X_test)

print("\n==============================")
print("ACCURACY")
print("==============================")
print(f"{accuracy_score(y_test, pred)*100:.2f}%")

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")
print(classification_report(y_test, pred, zero_division=0))

print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")
print(confusion_matrix(
    y_test,
    pred,
    labels=model.classes_
))

# -----------------------------
# FEATURE IMPORTANCE
# -----------------------------
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n==============================")
print("TOP FEATURES")
print("==============================")
print(importance)

# -----------------------------
# SAVE MODEL
# -----------------------------
joblib.dump(model, "multiclass_noc_attack_detector.pkl")
joblib.dump(features, "multiclass_noc_feature_names.pkl")

print("\nMODEL SAVED:")
print("multiclass_noc_attack_detector.pkl")
print("multiclass_noc_feature_names.pkl")

In [ ]:
from google.colab import files

files.download("multiclass_noc_attack_detector.pkl")
files.download("multiclass_noc_feature_names.pkl")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import joblib

model = joblib.load("multiclass_noc_attack_detector.pkl")
features = joblib.load("multiclass_noc_feature_names.pkl")

print("MODEL:", type(model).__name__)
print("FEATURES:", features)
print("CLASSES:", model.classes_)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# ============================================================
# FINAL — PROFESSIONAL NoC SECURITY DASHBOARD
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# CLOSE OLD APP
# ------------------------------------------------------------
try:
    demo.close()
except:
    pass


# ============================================================
# CSS
# ============================================================

CSS = """

/* ---------- PAGE ---------- */

.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
    background: #f6f8fb !important;
    font-family: Inter, Arial, sans-serif !important;
}

footer {
    display: none !important;
}


/* ---------- HEADER ---------- */

.header {
    background: #0b1220;
    color: white;
    border-radius: 18px;
    padding: 30px 34px;
    margin-bottom: 18px;
}

.header-title {
    font-size: 30px;
    font-weight: 800;
    letter-spacing: -0.8px;
}

.header-sub {
    margin-top: 7px;
    color: #9ca9bd;
    font-size: 14px;
}

.header-meta {
    margin-top: 18px;
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
}

.meta {
    border: 1px solid #26344d;
    background: #121c30;
    padding: 6px 10px;
    border-radius: 7px;
    color: #a9c7f7;
    font-size: 11px;
}


/* ---------- CARDS ---------- */

.card {
    background: white;
    border: 1px solid #e5e9f0;
    border-radius: 16px;
    padding: 22px;
    box-shadow: 0 5px 18px rgba(15,23,42,.04);
}

.card-title {
    font-size: 17px;
    font-weight: 750;
    color: #111827;
}

.card-sub {
    color: #7b8494;
    font-size: 12px;
    margin-top: 4px;
    margin-bottom: 18px;
}


/* ---------- DETECTION ---------- */

.detection {
    min-height: 180px;
    background: #0f172a;
    color: white;
    border-radius: 16px;
    padding: 25px;
}

.detection-label {
    font-size: 10px;
    color: #7fa8e8;
    letter-spacing: 1.6px;
    font-weight: 700;
}

.detection-result {
    font-size: 30px;
    font-weight: 850;
    margin-top: 14px;
    letter-spacing: -.7px;
}

.detection-confidence {
    margin-top: 12px;
    color: #aebbd0;
    font-size: 14px;
}

.detection-confidence b {
    color: white;
}


/* ---------- STATUS DOT ---------- */

.status-line {
    display: flex;
    align-items: center;
    gap: 8px;
    margin-top: 18px;
    color: #b9c5d7;
    font-size: 12px;
}

.dot {
    width: 8px;
    height: 8px;
    border-radius: 50%;
    background: #22c55e;
}


/* ---------- PROBABILITIES ---------- */

.probability {
    margin-top: 16px;
}

.prob-header {
    display: flex;
    justify-content: space-between;
    margin-bottom: 7px;
    font-size: 12px;
    font-weight: 650;
    color: #374151;
}

.track {
    height: 8px;
    background: #edf0f4;
    border-radius: 99px;
    overflow: hidden;
}

.fill {
    height: 100%;
    border-radius: 99px;
    background: #2563eb;
}


/* ---------- INDICATORS ---------- */

.indicator {
    display: flex;
    justify-content: space-between;
    padding: 13px 0;
    border-bottom: 1px solid #edf0f3;
}

.indicator-name {
    font-family: monospace;
    font-size: 11px;
    color: #4b5563;
}

.indicator-value {
    font-size: 12px;
    font-weight: 750;
    color: #111827;
}


/* ---------- ASSESSMENT ---------- */

.assessment {
    background: #f8fafc;
    border-left: 4px solid #2563eb;
    border-radius: 10px;
    padding: 18px;
    color: #374151;
    font-size: 13px;
    line-height: 1.65;
}


/* ---------- MODEL INFO ---------- */

.model-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 10px;
}

.model-item {
    background: #f8fafc;
    border: 1px solid #edf0f4;
    border-radius: 10px;
    padding: 14px;
}

.model-label {
    color: #8993a4;
    font-size: 10px;
    text-transform: uppercase;
    letter-spacing: .5px;
}

.model-value {
    color: #111827;
    font-size: 14px;
    font-weight: 750;
    margin-top: 5px;
}


/* ---------- BUTTON ---------- */

.analyze-btn {
    border-radius: 10px !important;
    min-height: 46px !important;
    font-weight: 750 !important;
}


/* ---------- PIPELINE ---------- */

.pipeline {
    background: white;
    border: 1px solid #e5e9f0;
    border-radius: 14px;
    padding: 15px 20px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 18px;
}

.pipeline-step {
    text-align: center;
    font-size: 10px;
    font-weight: 650;
    color: #596579;
}

.pipeline-icon {
    font-size: 18px;
    margin-bottom: 3px;
}

.pipeline-arrow {
    color: #a7b0bf;
}


/* ---------- MOBILE ---------- */

@media(max-width: 800px) {

    .model-grid {
        grid-template-columns: repeat(2,1fr);
    }

    .pipeline {
        gap: 10px;
    }

    .header-title {
        font-size: 24px;
    }
}

"""


# ============================================================
# RESULT PARSER
# ============================================================

def extract_value(text, pattern, default=""):
    m = re.search(pattern, text, re.I | re.S)
    return m.group(1).strip() if m else default


# ============================================================
# PROFESSIONAL ANALYSIS WRAPPER
# ============================================================

def professional_analyze(file):

    # Use your EXISTING working function
    result = analyze_csv(file)

    detection = str(result[0])
    probabilities = str(result[1])
    indicators = str(result[2])
    gemini_text = str(result[3])

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    prediction = extract_value(
        detection,
        r'class="prediction">(.*?)</div>'
    )

    if not prediction:
        for label in [
            "NETWORK ATTACK",
            "NODE-27 ATTACK",
            "NORMAL"
        ]:
            if label in detection.upper():
                prediction = label
                break

    confidence = extract_value(
        detection,
        r'Confidence:\s*(?:<b>)?([\d.]+)%'
    )

    if not confidence:
        confidence = extract_value(
            detection,
            r'([\d.]+)%'
        )

    # --------------------------------------------------------
    # DETECTION CARD
    # --------------------------------------------------------

    is_normal = prediction.upper() == "NORMAL"

    status_text = (
        "Traffic pattern classified within learned normal behavior."
        if is_normal
        else "Anomalous traffic pattern detected by the classifier."
    )

    detection_html = f"""
    <div class="detection">

        <div class="detection-label">
            NETWORK SECURITY STATUS
        </div>

        <div class="detection-result">
            {prediction}
        </div>

        <div class="detection-confidence">
            Detection confidence:
            <b>{confidence}%</b>
        </div>

        <div class="status-line">
            <span class="dot"></span>
            {status_text}
        </div>

    </div>
    """

    # --------------------------------------------------------
    # MODEL INFO
    # --------------------------------------------------------

    model_html = """
    <div class="card">

        <div class="card-title">
            Model Configuration
        </div>

        <div class="card-sub">
            Configuration used for the current security assessment.
        </div>

        <div class="model-grid">

            <div class="model-item">
                <div class="model-label">Algorithm</div>
                <div class="model-value">Random Forest</div>
            </div>

            <div class="model-item">
                <div class="model-label">Features</div>
                <div class="model-value">10</div>
            </div>

            <div class="model-item">
                <div class="model-label">Validation</div>
                <div class="model-value">5-Fold CV</div>
            </div>

            <div class="model-item">
                <div class="model-label">Samples</div>
                <div class="model-value">25</div>
            </div>

        </div>

    </div>
    """

    # --------------------------------------------------------
    # SHORT ASSESSMENT
    # --------------------------------------------------------

    if is_normal:

        assessment_text = """
        <b>Assessment:</b>
        The uploaded Noxim traffic is classified as
        <b>NORMAL</b>. Its observed network-performance
        characteristics are consistent with the normal
        samples used during model training.
        """

    elif "NODE-27" in prediction.upper():

        assessment_text = """
        <b>Assessment:</b>
        The uploaded Noxim traffic is classified as a
        <b>NODE-27 ATTACK</b> scenario according to the
        trained classifier. This indicates behavior closer
        to the malicious-node scenario represented in the
        training data.
        """

    else:

        assessment_text = """
        <b>Assessment:</b>
        The uploaded Noxim traffic is classified as a
        <b>NETWORK ATTACK</b> scenario according to the
        trained classifier. The decision is based on the
        learned network-performance characteristics.
        """

    assessment_html = f"""
    <div class="card">

        <div class="card-title">
            Security Assessment
        </div>

        <div class="card-sub">
            Concise interpretation of the model output.
        </div>

        <div class="assessment">
            {assessment_text}
        </div>

    </div>
    """

    # --------------------------------------------------------
    # KEEP YOUR EXISTING PROBABILITY + INDICATOR OUTPUT
    # --------------------------------------------------------

    # Remove unnecessary Gemini/Threat Library output completely.

    return (
        detection_html,
        probabilities,
        indicators,
        assessment_html,
        model_html
    )


# ============================================================
# DASHBOARD
# ============================================================

with gr.Blocks(
    title="NoC Security AI"
) as demo:

    # HEADER

    gr.HTML("""
    <div class="header">

        <div class="header-title">
            🛡️ NoC Security AI
        </div>

        <div class="header-sub">
            Explainable ML-based Network-on-Chip Attack Detection
        </div>

        <div class="header-meta">
            <span class="meta">Noxim Simulation</span>
            <span class="meta">Random Forest</span>
            <span class="meta">10 Network Features</span>
            <span class="meta">5-Fold Validation</span>
        </div>

    </div>
    """)


    # PIPELINE

    gr.HTML("""
    <div class="pipeline">

        <div class="pipeline-step">
            <div class="pipeline-icon">📄</div>
            Noxim CSV
        </div>

        <div class="pipeline-arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">🧠</div>
            ML Model
        </div>

        <div class="pipeline-arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">📊</div>
            Confidence
        </div>

        <div class="pipeline-arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">🔎</div>
            Indicators
        </div>

        <div class="pipeline-arrow">→</div>

        <div class="pipeline-step">
            <div class="pipeline-icon">✓</div>
            Assessment
        </div>

    </div>
    """)


    # INPUT + DETECTION

    with gr.Row():

        with gr.Column():

            gr.HTML("""
            <div class="card">

                <div class="card-title">
                    Analyze Network Traffic
                </div>

                <div class="card-sub">
                    Upload a Noxim network simulation CSV.
                </div>

            </div>
            """)

            upload = gr.File(
                label="Noxim CSV",
                file_types=[".csv"],
                type="filepath"
            )

            analyze_btn = gr.Button(
                "🔍  Analyze Network",
                variant="primary",
                elem_classes="analyze-btn"
            )


        with gr.Column():

            detection_output = gr.HTML()


    # CLASSIFICATION + INDICATORS

    with gr.Row():

        with gr.Column():

            gr.HTML("""
            <div class="card-title">
                Classification Confidence
            </div>

            <div class="card-sub">
                Probability distribution from the trained classifier.
            </div>
            """)

            probability_output = gr.HTML()


        with gr.Column():

            gr.HTML("""
            <div class="card-title">
                Key Network Indicators
            </div>

            <div class="card-sub">
                Model feature importance for the current analysis.
            </div>
            """)

            indicator_output = gr.HTML()


    # ASSESSMENT

    gr.HTML("""
    <div style="margin-top:22px;">
        <div class="card-title">
            Security Assessment
        </div>
    </div>
    """)

    assessment_output = gr.HTML()


    # MODEL

    gr.HTML("""
    <div style="margin-top:22px;">
        <div class="card-title">
            Model Information
        </div>
    </div>
    """)

    model_output = gr.HTML()


    # EVENT

    analyze_btn.click(
        fn=professional_analyze,
        inputs=upload,
        outputs=[
            detection_output,
            probability_output,
            indicator_output,
            assessment_output,
            model_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    theme=gr.themes.Base(),
    css=CSS
)

In [ ]:
# QUICK DIAGNOSTIC + TEST

import os
import pandas as pd
import joblib

print("MODEL FILE:", os.path.exists("multiclass_noc_attack_detector.pkl"))
print("FEATURE FILE:", os.path.exists("multiclass_noc_feature_names.pkl"))

model = joblib.load("multiclass_noc_attack_detector.pkl")
features = joblib.load("multiclass_noc_feature_names.pkl")

print("MODEL:", type(model).__name__)
print("FEATURES:", list(features))
print("CLASSES:", list(model.classes_))

# Find uploaded CSV
csvs = [x for x in os.listdir("/content") if x.endswith(".csv")]
print("CSV FILES:", csvs)

df = pd.read_csv("/content/BLACKHOLE_101.csv")

print("CSV SHAPE:", df.shape)
print("CSV COLUMNS:", list(df.columns))

X = df[list(features)].apply(pd.to_numeric, errors="coerce")

print("MISSING VALUES:", X.isnull().sum().sum())

prediction = model.predict(X)
probability = model.predict_proba(X)

print("================================")
print("PREDICTION:", prediction)
print("PROBABILITY:", probability)
print("================================")

In [ ]:
# FIX GRADIO APP

import gradio as gr
import pandas as pd
import joblib
import traceback

model = joblib.load("multiclass_noc_attack_detector.pkl")
features = joblib.load("multiclass_noc_feature_names.pkl")

def final_analyze(file):

    try:
        df = pd.read_csv(file)

        X = df[features].apply(pd.to_numeric, errors="coerce")

        pred = model.predict(X)[0]
        probs = model.predict_proba(X)[0]

        probability_html = ""

        for cls, p in zip(model.classes_, probs):
            probability_html += f"""
            <div style="margin:12px 0">
                <b>{cls}</b>
                <span style="float:right">{p*100:.2f}%</span>
                <div style="background:#eee;height:8px;border-radius:10px">
                    <div style="
                        width:{p*100}%;
                        background:#2563eb;
                        height:8px;
                        border-radius:10px">
                    </div>
                </div>
            </div>
            """

        evidence = ""

        for f in features:
            evidence += f"""
            <div style="
                display:flex;
                justify-content:space-between;
                padding:8px;
                border-bottom:1px solid #eee">
                <span>{f}</span>
                <b>{float(X.iloc[0][f]):.6f}</b>
            </div>
            """

        if pred == "NORMAL":
            assessment = "Network behavior is classified as NORMAL."
        elif pred == "DOS_FLOODING":
            assessment = "Potential DoS/Flooding behavior detected."
        elif pred == "BLACKHOLE":
            assessment = "Potential Blackhole attack behavior detected."
        elif pred == "ROUTING_MANIPULATION":
            assessment = "Potential Routing Manipulation behavior detected."
        else:
            assessment = "Unknown classification."

        detection = f"""
        <div style="
            background:#0f172a;
            color:white;
            padding:30px;
            border-radius:16px">

            <div style="color:#7fa8e8;font-size:12px">
                NETWORK SECURITY STATUS
            </div>

            <h1>{pred}</h1>

            <p>
                Detection confidence:
                <b>{max(probs)*100:.2f}%</b>
            </p>

            <p>● {assessment}</p>
        </div>
        """

        return (
            detection,
            probability_html,
            evidence,
            f"<div style='padding:20px'><b>Security Assessment</b><br><br>{assessment}</div>"
        )

    except Exception as e:

        traceback.print_exc()

        return (
            f"<div style='color:red'><b>Error:</b> {e}</div>",
            "",
            "",
            ""
        )


try:
    demo.close()
except:
    pass


with gr.Blocks(title="NoC Security AI") as demo:

    gr.Markdown("# 🛡️ NoC Security AI")
    gr.Markdown(
        "Explainable ML-based Network-on-Chip Attack Detection"
    )

    upload = gr.File(
        label="Upload Noxim CSV",
        file_types=[".csv"],
        type="filepath"
    )

    analyze = gr.Button(
        "🔍 Analyze Network",
        variant="primary"
    )

    detection = gr.HTML()

    gr.Markdown("### Classification Confidence")
    probabilities = gr.HTML()

    gr.Markdown("### Network Evidence")
    evidence = gr.HTML()

    gr.Markdown("### Security Assessment")
    assessment = gr.HTML()

    analyze.click(
        fn=final_analyze,
        inputs=upload,
        outputs=[
            detection,
            probabilities,
            evidence,
            assessment
        ]
    )

demo.launch(share=True)

In [ ]:
# =============================================================
# SECURE ROUTING LAYER
# AI ATTACK DETECTION -> MALICIOUS NODE ISOLATION -> SAFE ROUTE
# =============================================================

import networkx as nx
import pandas as pd
import numpy as np
import joblib
import os

print("==============================================")
print("      AI-ENABLED SECURE ROUTING MODULE")
print("==============================================")

# ------------------------------------------------------------
# LOAD EXISTING MULTI-CLASS MODEL AND FEATURES
# ------------------------------------------------------------

MODEL_PATH = "/content/multiclass_noc_attack_detector.pkl"
FEATURE_PATH = "/content/multiclass_noc_feature_names.pkl"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"{MODEL_PATH} not found. Please train/save the model first."
    )

model = joblib.load(MODEL_PATH)
features = joblib.load(FEATURE_PATH)

print("Model loaded successfully.")
print("Features expected:", features)
print("Classes:", list(model.classes_))

# ============================================================
# 1. CREATE WSN TOPOLOGY
# ============================================================

# 5 x 5 wireless sensor grid
ROWS = 5
COLS = 5

G = nx.grid_2d_graph(ROWS, COLS)

# Convert (row,column) -> integer node IDs
mapping = {
    node: node[0] * COLS + node[1]
    for node in G.nodes()
}

G = nx.relabel_nodes(G, mapping)

print("\nWSN topology created.")
print("Number of nodes:", len(G.nodes()))
print("Number of links:", len(G.edges()))

# ============================================================
# 2. SOURCE AND DESTINATION
# ============================================================

SOURCE_NODE = 0
DESTINATION_NODE = 24

print("Source node:", SOURCE_NODE)
print("Destination node:", DESTINATION_NODE)

# ============================================================
# 3. DETECT ATTACK FROM CSV
# ============================================================

def detect_attack(csv_file):
    df = pd.read_csv(csv_file)

    # Ensure we only use the exact features required by the loaded model
    missing = [f for f in features if f not in df.columns]
    if missing:
        raise ValueError(
            "CSV is missing model features:\n" + str(missing)
        )

    X = df[features].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    X = X.fillna(0)

    # Predict
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)

    # Majority prediction
    prediction_counts = pd.Series(predictions).value_counts()
    final_prediction = prediction_counts.index[0]

    # Average confidence
    class_index = list(model.classes_).index(final_prediction)
    confidence = probabilities[:, class_index].mean() * 100

    return final_prediction, confidence, predictions

# ============================================================
# 4. SECURE ROUTE FUNCTION
# ============================================================

def find_secure_route(source, destination, malicious_nodes):
    secure_graph = G.copy()

    # Remove malicious nodes
    for node in malicious_nodes:
        if node in secure_graph.nodes:
            # Never remove source or destination
            if node != source and node != destination:
                secure_graph.remove_node(node)

    try:
        route = nx.shortest_path(
            secure_graph,
            source=source,
            target=destination
        )
        return route
    except nx.NetworkXNoPath:
        return None

# ============================================================
# 5. ROUTING METRICS
# ============================================================

def calculate_route_metrics(route):
    if route is None:
        return {
            "status": "NO SAFE ROUTE",
            "hops": None,
            "packet_delivery_ratio": 0,
            "packet_loss": 100,
            "delay": None
        }

    hops = len(route) - 1
    packet_delivery_ratio = max(0, 100 - (hops * 1.5))
    packet_loss = 100 - packet_delivery_ratio
    delay = hops * 10

    return {
        "status": "SAFE ROUTE FOUND",
        "hops": hops,
        "packet_delivery_ratio": packet_delivery_ratio,
        "packet_loss": packet_loss,
        "delay": delay
    }

# ============================================================
# 6. MAIN SECURE ROUTING DEMO
# ============================================================

def secure_routing_demo(csv_file, malicious_node):
    prediction, confidence, row_predictions = detect_attack(csv_file)

    print("\n==============================================")
    print("              AI DETECTION")
    print("==============================================")
    print("Prediction :", prediction)
    print(f"Confidence : {confidence:.2f}%")

    print("\nClass distribution:")
    print(pd.Series(row_predictions).value_counts())

    if str(prediction).upper() == "NORMAL":
        malicious_nodes = []
        print("\nSTATUS: NORMAL")
        print("No malicious node isolated.")
    else:
        malicious_nodes = [malicious_node]
        print("\nSTATUS: ATTACK DETECTED")
        print("Attack type:", prediction)
        print("Malicious node:", malicious_node)
        print("Action: Node isolated from routing")

    # ORIGINAL ROUTE
    original_route = nx.shortest_path(
        G,
        source=SOURCE_NODE,
        target=DESTINATION_NODE
    )

    print("\n==============================================")
    print("              ORIGINAL ROUTE")
    print("==============================================")
    print(" -> ".join(map(str, original_route)))

    # SECURE ROUTE
    secure_route = find_secure_route(
        SOURCE_NODE,
        DESTINATION_NODE,
        malicious_nodes
    )

    print("\n==============================================")
    print("               SECURE ROUTE")
    print("==============================================")
    if secure_route:
        print(" -> ".join(map(str, secure_route)))
    else:
        print("No secure route available.")

    original_metrics = calculate_route_metrics(original_route)
    secure_metrics = calculate_route_metrics(secure_route)

    results = pd.DataFrame({
        "Metric": [
            "Route",
            "Hops",
            "Packet Delivery Ratio (%)",
            "Packet Loss (%)",
            "Delay"
        ],
        "Normal/Original": [
            " -> ".join(map(str, original_route)),
            original_metrics["hops"],
            round(original_metrics["packet_delivery_ratio"], 2),
            round(original_metrics["packet_loss"], 2),
            original_metrics["delay"]
        ],
        "Secure Routing": [
            " -> ".join(map(str, secure_route)) if secure_route else "NO ROUTE",
            secure_metrics["hops"],
            round(secure_metrics["packet_delivery_ratio"], 2),
            round(secure_metrics["packet_loss"], 2),
            secure_metrics["delay"]
        ]
    })

    print("\n==============================================")
    print("           ROUTING PERFORMANCE")
    print("==============================================")
    display(results)

    return {
        "prediction": prediction,
        "confidence": confidence,
        "malicious_nodes": malicious_nodes,
        "original_route": original_route,
        "secure_route": secure_route,
        "results": results
    }

In [ ]:
result = secure_routing_demo(
    "/content/DOS_FLOODING_101.csv",
    malicious_node=12
)

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib
import os
import glob
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# AUTO-GENERATE MODEL IF MISSING
# ------------------------------------------------------------

MODEL_PATH = "/content/multiclass_noc_attack_detector.pkl"
FEATURE_PATH = "/content/multiclass_noc_feature_names.pkl"

# Fallback paths
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = "multiclass_noc_attack_detector.pkl"
if not os.path.exists(FEATURE_PATH):
    FEATURE_PATH = "multiclass_noc_feature_names.pkl"

if not os.path.exists(MODEL_PATH) or not os.path.exists(FEATURE_PATH):
    print("Model files not found! Automatically training a new multi-class model from uploaded CSVs...")
    csv_files = glob.glob("/content/*.csv")
    rows = []
    for name in csv_files:
        filename = os.path.basename(name).upper()
        if "NORMAL" in filename:
            label = "NORMAL"
        elif "DOS_FLOODING" in filename:
            label = "DOS_FLOODING"
        elif "BLACKHOLE" in filename:
            label = "BLACKHOLE"
        elif "ROUTING_MANIPULATION" in filename:
            label = "ROUTING_MANIPULATION"
        else:
            continue
        try:
            df_temp = pd.read_csv(name)
            df_temp["label"] = label
            rows.append(df_temp)
        except:
            continue

    if not rows:
        raise FileNotFoundError("Could not find any labeled CSV files to train the model. Please upload them first.")

    data = pd.concat(rows, ignore_index=True)
    features = [
        "total_received_packets",
        "total_received_flits",
        "received_ideal_flits_ratio",
        "global_average_delay_cycles",
        "max_delay_cycles",
        "network_throughput_flits_per_cycle",
        "average_ip_throughput_flits_per_cycle_per_ip",
        "total_energy_j",
        "dynamic_energy_j",
        "static_energy_j"
    ]
    X = data[features].copy()
    y = data["label"].copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    model.fit(X, y)

    MODEL_PATH = "/content/multiclass_noc_attack_detector.pkl"
    FEATURE_PATH = "/content/multiclass_noc_feature_names.pkl"
    joblib.dump(model, MODEL_PATH)
    joblib.dump(features, FEATURE_PATH)
    print("Model auto-trained and saved successfully!")

model = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURE_PATH)

print("MODEL LOADED:", type(model).__name__)
print("FEATURES LOADED:", feature_names)
print("CLASSES:", model.classes_)

# ------------------------------------------------------------
# ANALYSIS FUNCTION
# ------------------------------------------------------------

def analyze_csv(file):
    if file is None:
        raise ValueError("Please upload a Noxim CSV file.")

    # Read CSV
    df = pd.read_csv(file)

    # Check required features
    missing = [f for f in feature_names if f not in df.columns]
    if missing:
        raise ValueError("Missing required features: " + ", ".join(missing))

    # Use exactly the same feature order as training
    X = df[feature_names].copy()

    # Convert numeric
    X = X.apply(pd.to_numeric, errors="coerce")
    if X.isnull().any().any():
        X = X.fillna(0)

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------
    prediction = model.predict(X)[0]
    probabilities = model.predict_proba(X)[0]
    classes = list(model.classes_)
    prob_dict = dict(zip(classes, probabilities))
    confidence = float(max(probabilities) * 100)

    # --------------------------------------------------------
    # NETWORK EVIDENCE
    # --------------------------------------------------------
    row = X.iloc[0]
    evidence = {k: row[k] for k in feature_names if k in row}

    # --------------------------------------------------------
    # FEATURE IMPORTANCE
    # --------------------------------------------------------
    importances = model.feature_importances_
    importance_pairs = sorted(
        zip(feature_names, importances),
        key=lambda x: x[1],
        reverse=True
    )

    return prediction, confidence, prob_dict, evidence, importance_pairs

# ------------------------------------------------------------
# PROFESSIONAL HTML OUTPUT
# ------------------------------------------------------------

def professional_analyze(file):
    try:
        prediction, confidence, probs, evidence, importances = analyze_csv(file)
        prediction = str(prediction)
        is_normal = prediction == "NORMAL"

        if is_normal:
            status = "Normal network behavior detected."
        else:
            status = "Potential NoC attack behavior detected."

        detection_html = f"""
        <div style="
            background:#0f172a;
            color:white;
            border-radius:16px;
            padding:28px;
            margin-bottom:18px;
        ">
            <div style="
                color:#7fa8e8;
                font-size:11px;
                font-weight:700;
                letter-spacing:1.5px;
            ">
                NETWORK SECURITY STATUS
            </div>

            <div style="
                font-size:30px;
                font-weight:850;
                margin-top:14px;
            ">
                {prediction}
            </div>

            <div style="
                margin-top:12px;
                color:#aebbd0;
                font-size:14px;
            ">
                Detection confidence:
                <b style="color:white">{confidence:.2f}%</b>
            </div>

            <div style="
                margin-top:18px;
                color:#b9c5d7;
                font-size:12px;
            ">
                ● &nbsp; {status}
            </div>
        </div>
        """

        probability_html = """
        <div style="
            background:white;
            border:1px solid #e5e9f0;
            border-radius:16px;
            padding:22px;
        ">
        """
        for cls in model.classes_:
            p = float(probs[cls]) * 100
            probability_html += f"""
            <div style="margin-top:14px;">
                <div style="
                    display:flex;
                    justify-content:space-between;
                    font-size:12px;
                    font-weight:650;
                    color:#374151;
                    margin-bottom:7px;
                ">
                    <span>{cls}</span>
                    <span>{p:.2f}%</span>
                </div>
                <div style="
                    height:8px;
                    background:#edf0f4;
                    border-radius:99px;
                    overflow:hidden;
                ">
                    <div style="
                        width:{p}%;
                        height:100%;
                        background:#2563eb;
                        border-radius:99px;
                    "></div>
                </div>
            </div>
            """
        probability_html += "</div>"

        indicator_html = """
        <div style="
            background:white;
            border:1px solid #e5e9f0;
            border-radius:16px;
            padding:22px;
        ">
        """
        display_names = {
            "global_average_delay_cycles": "Average Delay",
            "max_delay_cycles": "Max Delay",
            "total_received_packets": "Received Packets",
            "total_received_flits": "Received Flits",
            "received_ideal_flits_ratio": "Ideal Flits Ratio",
            "network_throughput_flits_per_cycle": "Network Throughput",
            "average_ip_throughput_flits_per_cycle_per_ip": "IP Throughput",
            "total_energy_j": "Total Energy",
            "dynamic_energy_j": "Dynamic Energy",
            "static_energy_j": "Static Energy"
        }

        for key, value in evidence.items():
            name = display_names.get(key, key)
            try:
                value_text = f"{(float(value)):.6f}"
            except:
                value_text = str(value)

            indicator_html += f"""
            <div style="
                display:flex;
                justify-content:space-between;
                padding:11px 0;
                border-bottom:1px solid #edf0f3;
            ">
                <span style="
                    font-family:monospace;
                    font-size:11px;
                    color:#4b5563;
                ">
                    {name}
                </span>
                <span style="
                    font-size:12px;
                    font-weight:750;
                    color:#111827;
                ">
                    {value_text}
                </span>
            </div>
            """
        indicator_html += "</div>"

        importance_html = """
        <div style="
            background:white;
            border:1px solid #e5e9f0;
            border-radius:16px;
            padding:22px;
            margin-top:18px;
        ">
            <div style="
                font-size:17px;
                font-weight:750;
                color:#111827;
            ">
                Model Indicators
            </div>
            <div style="
                color:#7b8494;
                font-size:12px;
                margin-top:4px;
                margin-bottom:15px;
            ">
                Features with the highest learned importance in the Random Forest.
            </div>
        """
        for name, importance in importances[:5]:
            importance_html += f"""
            <div style="
                display:flex;
                justify-content:space-between;
                padding:10px 0;
                border-bottom:1px solid #edf0f3;
            ">
                <span style="
                    font-family:monospace;
                    font-size:11px;
                    color:#4b5563;
                ">
                    {name}
                </span>
                <b style="font-size:12px;">
                    {importance:.4f}
                </b>
            </div>
            """
        importance_html += "</div>"

        if prediction == "NORMAL":
            assessment = "The uploaded Noxim traffic is classified as <b>NORMAL</b>. Its observed characteristics align with training baseline traffic."
        elif prediction == "DOS_FLOODING":
            assessment = "The traffic is classified as <b>DOS_FLOODING</b>. The model has identified high congestion and flit rates indicating DoS flooding behavior."
        elif prediction == "BLACKHOLE":
            assessment = "The traffic is classified as <b>BLACKHOLE</b>. Packet losses and abnormal drop ratios signify simulated blackhole activity."
        elif prediction == "ROUTING_MANIPULATION":
            assessment = "The traffic is classified as <b>ROUTING_MANIPULATION</b>. Packet paths deviate considerably from optimal network expectations."
        else:
            assessment = f"The classifier predicted <b>{prediction}</b>."

        assessment_html = f"""
        <div style="
            background:white;
            border:1px solid #e5e9f0;
            border-radius:16px;
            padding:22px;
            margin-top:18px;
        ">
            <div style="
                font-size:17px;
                font-weight:750;
                color:#111827;
                margin-bottom:14px;
            ">
                Security Assessment
            </div>
            <div style="
                background:#f8fafc;
                border-left:4px solid #2563eb;
                border-radius:10px;
                padding:18px;
                color:#374151;
                font-size:13px;
                line-height:1.65;
            ">
                {assessment}
            </div>
        </div>
        """

        model_html = f"""
        <div style="
            background:white;
            border:1px solid #e5e9f0;
            border-radius:16px;
            padding:22px;
            margin-top:18px;
        ">
            <div style="
                font-size:17px;
                font-weight:750;
                color:#111827;
            ">
                Model Information
            </div>
            <div style="
                color:#7b8494;
                font-size:12px;
                margin-top:4px;
                margin-bottom:18px;
            ">
                Configuration used for the current security assessment.
            </div>
            <div style="
                display:grid;
                grid-template-columns:repeat(4,1fr);
                gap:10px;
            ">
                <div style="background:#f8fafc;padding:14px;border-radius:10px;">
                    <small>ALGORITHM</small><br>
                    <b>Random Forest</b>
                </div>
                <div style="background:#f8fafc;padding:14px;border-radius:10px;">
                    <small>FEATURES</small><br>
                    <b>{len(feature_names)}</b>
                </div>
                <div style="background:#f8fafc;padding:14px;border-radius:10px;">
                    <small>CLASSES</small><br>
                    <b>{len(model.classes_)}</b>
                </div>
                <div style="background:#f8fafc;padding:14px;border-radius:10px;">
                    <small>DATASET</small><br>
                    <b>Multi-class Baseline</b>
                </div>
            </div>
        </div>
        """
        return (
            detection_html,
            probability_html,
            indicator_html + importance_html,
            assessment_html,
            model_html
        )
    except Exception as e:
        return (
            f"<div style='color:red;padding:20px;'><b>Analysis Error:</b> {str(e)}</div>",
            "",
            "",
            "",
            ""
        )

# ------------------------------------------------------------
# CLOSE OLD APP IF EXISTS
# ------------------------------------------------------------

try:
    demo.close()
except:
    pass

# ------------------------------------------------------------
# NEW DASHBOARD
# ------------------------------------------------------------

with gr.Blocks(title="NoC Security AI") as demo:
    gr.HTML("""
    <div style="
        background:#0b1220;
        color:white;
        border-radius:18px;
        padding:30px 34px;
        margin-bottom:18px;
    ">
        <div style="font-size:30px;font-weight:800;">
            🛡️ NoC Security AI
        </div>
        <div style="
            margin-top:7px;
            color:#9ca9bd;
            font-size:14px;
        ">
            Explainable ML-based Network-on-Chip Attack Detection
        </div>
        <div style="
            margin-top:18px;
            display:flex;
            gap:8px;
            flex-wrap:wrap;
        ">
            <span style="border:1px solid #26344d;background:#121c30;padding:6px 10px;border-radius:7px;color:#a9c7f7;font-size:11px;">Noxim Simulation</span>
            <span style="border:1px solid #26344d;background:#121c30;padding:6px 10px;border-radius:7px;color:#a9c7f7;font-size:11px;">Random Forest</span>
            <span style="border:1px solid #26344d;background:#121c30;padding:6px 10px;border-radius:7px;color:#a9c7f7;font-size:11px;">10 Network Features</span>
            <span style="border:1px solid #26344d;background:#121c30;padding:6px 10px;border-radius:7px;color:#a9c7f7;font-size:11px;">4 Attack Classes</span>
        </div>
    </div>
    """)

    gr.HTML("""
    <div style="
        background:white;
        border:1px solid #e5e9f0;
        border-radius:14px;
        padding:15px 20px;
        display:flex;
        justify-content:space-between;
        margin-bottom:18px;
        font-size:11px;
        font-weight:650;
        color:#596579;
    ">
        📄 Noxim CSV
        <span>→</span>
        🧠 Random Forest
        <span>→</span>
        📊 Confidence
        <span>→</span>
        🔎 Network Evidence
        <span>→</span>
        ✓ Assessment
    </div>
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Analyze Network Traffic")
            gr.Markdown("Upload a Noxim network simulation CSV.")
            upload = gr.File(label="Noxim CSV", file_types=[".csv"], type="filepath")
            analyze_btn = gr.Button("🔍 Analyze Network", variant="primary")
        with gr.Column():
            detection_output = gr.HTML()

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Classification Confidence")
            probability_output = gr.HTML()
        with gr.Column():
            gr.Markdown("### Key Network Indicators")
            indicator_output = gr.HTML()

    assessment_output = gr.HTML()
    model_output = gr.HTML()

    analyze_btn.click(
        fn=professional_analyze,
        inputs=upload,
        outputs=[
            detection_output,
            probability_output,
            indicator_output,
            assessment_output,
            model_output
        ]
    )

demo.launch(share=True)

In [ ]:
import pandas as pd
import io
from google.colab import files

# Upload all 20 simulation CSV files
uploaded = files.upload()

# Combine all files
dataframes = []

for filename, content in uploaded.items():
    df = pd.read_csv(io.BytesIO(content))

    # Get class from filename, e.g. BLACKHOLE_101.csv
    attack_label = filename.split("_")[0].upper()

    df["attack_label"] = attack_label
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)

# Save combined dataset
combined_df.to_csv("combined_noc_dataset.csv", index=False)

print("Total records:", len(combined_df))
print("\nClass distribution:")
print(combined_df["attack_label"].value_counts())

files.download("combined_noc_dataset.csv")

In [ ]:
import pandas as pd
import glob
import os

files = glob.glob("/content/*.csv")
dataframes = []

def get_label(filename):
    name = os.path.basename(filename).lower()

    if "routing" in name:
        return "ROUTING_MANIPULATION"
    elif "blackhole" in name:
        return "BLACKHOLE"
    elif "dos" in name or name.startswith("attack"):
        return "DOS_FLOODING"
    elif "normal" in name:
        return "NORMAL"
    return None

for filename in files:
    label = get_label(filename)
    if label is None:
        continue

    df = pd.read_csv(filename)
    df["attack_label"] = label
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)
combined_df.to_csv("/content/combined_noc_dataset.csv", index=False)

print("Total records:", len(combined_df))
print("\nClass distribution:")
print(combined_df["attack_label"].value_counts())

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # Select both .pkl files

print("Uploaded files:")
print(os.listdir("/content"))

In [ ]:
import os

for f in os.listdir("/content"):
    if f.endswith(".pkl"):
        print(f)

In [ ]:
import joblib
from google.colab import files

joblib.dump(model, "/content/multiclass_noc_attack_detector.pkl")
joblib.dump(features, "/content/multiclass_noc_feature_names.pkl")

files.download("/content/multiclass_noc_attack_detector.pkl")
files.download("/content/multiclass_noc_feature_names.pkl")

In [ ]:
print("Available model-related variables:")
print([name for name in globals() if "model" in name.lower() or "feature" in name.lower()])

In [ ]:
import pandas as pd
import glob, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv("/content/combined_noc_dataset.csv")

features = [
    "total_received_packets",
    "total_received_flits",
    "received_ideal_flits_ratio",
    "global_average_delay_cycles",
    "max_delay_cycles",
    "network_throughput_flits_per_cycle",
    "average_ip_throughput_flits_per_cycle_per_ip",
    "total_energy_j",
    "dynamic_energy_j",
    "static_energy_j"
]

X = df[features].apply(pd.to_numeric, errors="coerce")
y = df["attack_label"]

valid = X.notna().all(axis=1) & y.notna()
X, y = X[valid], y[valid]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

joblib.dump(model, "/content/multiclass_noc_attack_detector.pkl")
joblib.dump(features, "/content/multiclass_noc_feature_names.pkl")

print("Model saved successfully.")
print("Classes:", model.classes_)
print("Training records:", len(X))

In [ ]:
import pandas as pd
from google.colab import files

df = pd.read_csv("/content/combined_noc_dataset.csv")
X = df[features].apply(pd.to_numeric, errors="coerce")

df["predicted_attack"] = model.predict(X)

print("PREDICTED ATTACK DISTRIBUTION")
print(
    df["predicted_attack"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .astype(str)
    .add("%")
)

df.to_csv("/content/noc_predictions.csv", index=False)
files.download("/content/noc_predictions.csv")

In [ ]:
import gradio as gr
import pandas as pd

df = pd.read_csv("/content/noc_predictions.csv")

classes = [
    "NORMAL",
    "DOS_FLOODING",
    "BLACKHOLE",
    "ROUTING_MANIPULATION"
]

counts = df["predicted_attack"].value_counts().reindex(classes, fill_value=0)

summary = pd.DataFrame({
    "Attack Class": classes,
    "Records": counts.values,
    "Percentage": (counts.values / len(df) * 100).round(2)
})

with gr.Blocks(theme=gr.themes.Soft(), title="NoC Security Analytics") as demo:
    gr.Markdown("# NoC Security Analytics")
    gr.Markdown("### Explainable ML-Based Network Attack Detection")
    gr.Markdown(f"**Records analyzed:** {len(df)}")

    with gr.Row():
        for cls in classes:
            pct = summary.loc[summary["Attack Class"] == cls, "Percentage"].iloc[0]
            gr.Number(value=float(pct), label=f"{cls} (%)", interactive=False)

    gr.BarPlot(
        value=summary,
        x="Attack Class",
        y="Percentage",
        title="Predicted Attack Distribution",
        y_lim=[0, 100]
    )

    gr.Dataframe(value=summary, label="Detection Summary", interactive=False)

    gr.Markdown(
        "Note: This is a prototype trained on 49 records. "
        "Predictions and percentages require independent validation."
    )

demo.launch(share=True)

In [ ]:
import gradio as gr

def inspect_node(node, attack):
    x, y = node % 4, node // 4

    plans = {
        "BLACKHOLE": "Isolate suspected node and test an alternative route.",
        "DOS_FLOODING": "Reduce injection load and test network recovery.",
        "ROUTING_MANIPULATION": "Use trusted routing and validate the new path.",
        "NORMAL": "No mitigation required."
    }

    status = (
        "No attack selected"
        if attack == "NORMAL"
        else "Mitigation plan proposed — simulation verification pending"
    )

    return (
        f"Node {node} (x={x}, y={y})",
        "Not measured — port-level telemetry required",
        plans[attack],
        status
    )

with gr.Blocks(theme=gr.themes.Soft(), title="NoC Mitigation Monitor") as mitigation:
    gr.Markdown("# NoC Mitigation Monitor")
    gr.Markdown("### 4×4 Network-on-Chip | Nodes 0–15")

    gr.Markdown(
        "**Node layout**\n\n"
        "12 · 13 · 14 · 15\n\n"
        "8 · 9 · 10 · 11\n\n"
        "4 · 5 · 6 · 7\n\n"
        "0 · 1 · 2 · 3"
    )

    node = gr.Dropdown(
        choices=list(range(16)),
        value=6,
        label="Select suspected node"
    )

    attack = gr.Dropdown(
        choices=[
            "NORMAL", "DOS_FLOODING",
            "BLACKHOLE", "ROUTING_MANIPULATION"
        ],
        value="BLACKHOLE",
        label="Attack class"
    )

    inspect = gr.Button("Inspect and Plan Mitigation")

    node_info = gr.Textbox(label="Selected node")
    port_info = gr.Textbox(label="Port availability")
    plan_info = gr.Textbox(label="Proposed mitigation")
    status_info = gr.Textbox(label="Security status")

    inspect.click(
        inspect_node,
        inputs=[node, attack],
        outputs=[node_info, port_info, plan_info, status_info]
    )

mitigation.launch(share=True)

In [ ]:
from pathlib import Path
import subprocess
import os

ROOT = Path("/content/noxim")
assert (ROOT / "src/Router.cpp").exists(), "Project source missing."

# Use the SystemC library already installed in Colab.
libs = list(Path("/usr/lib").glob("*/libsystemc.so"))
libs += list(Path("/lib").glob("*/libsystemc.so"))
if not libs:
    raise RuntimeError("Colab SystemC library not found.")

libdir = libs[0].parent
makefile = ROOT / "bin/Makefile"
lines = makefile.read_text().splitlines()

for i, line in enumerate(lines):
    if line.startswith("SYSTEMC :="):
        lines[i] = "SYSTEMC := /usr"
    elif line.startswith("SYSTEMC_LIBS :="):
        lines[i] = f"SYSTEMC_LIBS := {libdir}"

makefile.write_text("\n".join(lines) + "\n")

print("Building with:", libdir)
build = subprocess.run(
    ["make", "-C", str(ROOT / "bin"), "-B", "-j2"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print(build.stdout[-5000:])

if build.returncode != 0:
    raise RuntimeError("Build failed. Stop here and send the error.")

binary = ROOT / "bin/noxim"
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    f"{libdir}:{ROOT / 'bin/libs/yaml-cpp/lib'}:"
    + env.get("LD_LIBRARY_PATH", "")
)
for key in ("NOXIM_ATTACK", "NOXIM_MAL_NODE", "NOXIM_MITIGATE"):
    env.pop(key, None)

print("Running a short normal-traffic test...")
run = subprocess.run(
    [str(binary), "-config", "config_examples/default_configMesh.yaml",
     "-power", "bin/power.yaml", "-sim", "3000", "-warmup", "1000",
     "-seed", "101", "-pir", "0.01", "poisson", "-routing", "XY",
     "-stats_format", "csv",
     "-stats_file", str(ROOT / "colab_smoke_test.csv")],
    cwd=ROOT, env=env, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, timeout=120
)
print("\n".join(
    line for line in run.stdout.splitlines()
    if any(k in line for k in (
        "Noxim simulation completed", "Total received packets:",
        "Received/Ideal flits Ratio:", "Global average delay"
    ))
))
if run.returncode != 0:
    raise RuntimeError("Simulation test failed. Send me the output.")

print("✅ Build and normal-traffic smoke test passed.")

AssertionError: Project source missing.

In [ ]:
import os
from pathlib import Path

print("=== Checking Current Colab Environment ===")
print(f"Current working directory: {os.getcwd()}")

paths_to_check = [
    "/content/noxim_colab_bundle.tar.gz",
    "/content/noxim",
    "/content/noxim/src/Router.cpp",
    "/content/multiclass_noc_attack_detector.pkl",
    "/content/network_attack_detector.pkl",
    "/content/feature_names.pkl"
]

for p in paths_to_check:
    path = Path(p)
    print(f"{'[FOUND]' if path.exists() else '[MISSING]'}  {p}")

print("\n=== Files in /content ===")
for item in sorted(os.listdir('/content')):
    print(f" - {item}")

MISSING  /content/noxim
MISSING  /content/multiclass_noc_attack_detector.pkl
MISSING  /content/noxim/network_attack_detector.pkl
MISSING  /content/noxim/feature_names.pkl


In [ ]:
%%bash
echo "===== 1. OS / glibc ====="
cat /etc/os-release | grep -E "PRETTY_NAME|VERSION_ID"
ldd --version | head -1
echo
echo "===== 2. Compiler / tools ====="
g++ --version | head -1
make --version | head -1
python3 --version
echo
echo "===== 3. /content listing ====="
ls -la /content
echo
echo "===== 4. Archive present? ====="
find /content -maxdepth 2 \( -name "*.tar.gz" -o -name "*.tgz" \) -exec ls -lh {} \; 2>/dev/null
echo
echo "===== 5. Project folder ====="
if [ -d /content/noxim ]; then
  echo "/content/noxim EXISTS"
  ls -la /content/noxim
  echo "--- Router.cpp / ProcessingElement.cpp ---"
  ls -l /content/noxim/src/Router.cpp /content/noxim/src/ProcessingElement.cpp 2>&1
  echo "--- key project files ---"
  ls -l /content/noxim/app.py /content/noxim/*.pkl /content/noxim/*.csv /content/noxim/build.sh 2>&1
else
  echo "/content/noxim DOES NOT EXIST in this runtime"
fi
echo
echo "===== 6. Any other copy of Router.cpp on disk? ====="
find / -name "Router.cpp" -not -path "*/proc/*" 2>/dev/null | head
echo
echo "===== 7. SystemC installed via apt? ====="
dpkg -l 2>/dev/null | grep -i systemc || echo "no systemc package found"
echo
echo "===== 8. SystemC headers and libs on system ====="
find /usr -name "systemc" -maxdepth 4 -type d 2>/dev/null | head
find /usr /lib -name "libsystemc*" 2>/dev/null | head
echo
echo "===== 9. Existing noxim binary (if any) and what it needs ====="
find /content -name "noxim" -type f -perm -u+x 2>/dev/null | head -3
for b in $(find /content -name "noxim" -type f -perm -u+x 2>/dev/null | head -1); do
  echo "Checking $b"
  ldd "$b" 2>&1 | head -20
done

===== 1. OS / glibc =====
PRETTY_NAME="Ubuntu 24.04.5 LTS"
VERSION_ID="24.04"
ldd (Ubuntu GLIBC 2.39-0ubuntu8.9) 2.39

===== 2. Compiler / tools =====
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
GNU Make 4.3
Python 3.13.16

===== 3. /content listing =====
total 16
drwxr-xr-x 1 root root 4096 Oct  2 13:50 .
drwxr-xr-x 1 root root 4096 Oct 13 14:58 ..
drwxr-xr-x 4 root root 4096 Oct  2 13:50 .config
drwxr-xr-x 1 root root 4096 Oct  2 13:50 sample_data

===== 4. Archive present? =====

===== 5. Project folder =====
/content/noxim DOES NOT EXIST in this runtime

===== 6. Any other copy of Router.cpp on disk? =====

===== 7. SystemC installed via apt? =====
no systemc package found

===== 8. SystemC headers and libs on system =====

===== 9. Existing noxim binary (if any) and what it needs =====


In [ ]:
import os, subprocess, shutil, glob

PROJ   = "/content/noxim"
ROUTER = f"{PROJ}/src/Router.cpp"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")

# ---------- PART 1: upload + extract (skipped if already present) ----------
if os.path.exists(ROUTER):
    print("[1] Project already present, skipping upload.")
else:
    print("[1] Checking for uploaded archive...")
    tars = glob.glob("/content/*.tar.gz") + glob.glob("/content/*.tgz")
    if not tars:
        print("⚠️ No .tar.gz or .tgz archive found in /content. Skipping C++ compilation/extraction steps.")
        print("Proceeding with simulation dataset analysis instead.")
    else:
        tar = tars[0]
        shutil.rmtree("/content/_extract", ignore_errors=True)
        os.makedirs("/content/_extract")
        r = sh(f"tar -xzf '{tar}' -C /content/_extract")
        assert r.returncode == 0, "tar failed:\n" + r.stderr[-1500:]
        hits = glob.glob("/content/_extract/**/src/Router.cpp", recursive=True)
        assert hits, "Router.cpp not found inside archive"
        root = os.path.dirname(os.path.dirname(hits[0]))
        if os.path.exists(PROJ):
            shutil.move(PROJ, "/content/noxim_old_partial")
        shutil.move(root, PROJ)
        print(f"[1] Extracted to {PROJ}")

# ---------- PART 2: SystemC from apt, with logs captured ----------
r1 = sh("apt-get update > /content/apt_update.log 2>&1; echo $?")
r2 = sh("DEBIAN_FRONTEND=noninteractive apt-get install -y libsystemc-dev > /content/apt_install.log 2>&1; echo $?")
print(f"\n[2] apt-get update exit = {r1.stdout.strip()} | install exit = {r2.stdout.strip()}")
if r2.stdout.strip() != "0":
    print("---- last 25 lines of apt_install.log ----")
    print(sh("tail -25 /content/apt_install.log").stdout)

# ---------- PART 3: read-only facts needed for the build fix ----------
print("\n[3] Project files")
if os.path.exists(PROJ):
    print(sh(f"ls -l {PROJ}/build.sh {PROJ}/bin/Makefile {PROJ}/app.py {PROJ}/*.pkl {PROJ}/node6_attack_comparison.csv 2>&1").stdout)
else:
    print("/content/noxim folder not present.")

csv_files = glob.glob('/content/*.csv')
print("CSV count in /content:", len(csv_files))
print("Prebuilt binary present:", os.path.exists(f"{PROJ}/bin/noxim") if os.path.exists(PROJ) else False)

print("\n[4] SystemC on this system")
print(sh("dpkg -l | grep -i systemc; echo ---; find /usr -name 'systemc.h' 2>/dev/null; "
         "find /usr -name 'libsystemc*' 2>/dev/null").stdout)

if os.path.exists(PROJ):
    print("[5] bin/Makefile (SystemC / flags / paths)")
    print(sh(f"grep -nE 'SYSTEMC|CXX|CXXFLAGS|CPPFLAGS|LDFLAGS|LIBS|-std|-I|-L|rpath' {PROJ}/bin/Makefile | head -60").stdout)
    print("[6] build.sh")
    print(sh(f"cat {PROJ}/build.sh").stdout)
else:
    print("[5/6] Skipping Makefile/build.sh audits (no project folder).")

print("nproc:", sh("nproc").stdout.strip())

[1] Checking for uploaded archive...
⚠️ No .tar.gz or .tgz archive found in /content. Skipping C++ compilation/extraction steps.
Proceeding with simulation dataset analysis instead.

[2] apt-get update exit = 0 | install exit = 0

[3] Project files
/content/noxim folder not present.
CSV count in /content: 50
Prebuilt binary present: False

[4] SystemC on this system
ii  libsystemc:amd64                   2.3.4-3build1                        amd64        SystemC library
ii  libsystemc-dev:amd64               2.3.4-3build1                        amd64        Development files for SystemC library
---
/usr/include/systemc.h
/usr/share/doc/libsystemc-dev
/usr/share/doc/libsystemc
/usr/share/lintian/overrides/libsystemc-dev
/usr/share/lintian/overrides/libsystemc
/usr/lib/x86_64-linux-gnu/libsystemc.so
/usr/lib/x86_64-linux-gnu/libsystemc-2.3.4.so

[5/6] Skipping Makefile/build.sh audits (no project folder).
nproc: 2


In [ ]:
import os, subprocess, shutil, glob

PROJ   = "/content/noxim"
ROUTER = f"{PROJ}/src/Router.cpp"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")

if os.path.exists(ROUTER):
    print("Project already present, skipping upload.")
else:
    from google.colab import files
    print("Please upload a Noxim source archive (.tar.gz/.tgz) if you wish to compile C++ source, or cancel if you only wish to analyze CSVs:")
    try:
        up = files.upload()
        tars = [k for k in up if k.endswith((".tar.gz", ".tgz"))]

        if not tars:
            print("\n⚠️ No .tar.gz or .tgz archive was uploaded (only CSVs or other files found).")
            print("Skipping C++ extraction/compilation and proceeding directly with simulation dataset analysis.")
        else:
            shutil.rmtree("/content/_extract", ignore_errors=True)
            os.makedirs("/content/_extract")
            r = sh(f"tar -xzf '/content/{tars[0]}' -C /content/_extract")
            assert r.returncode == 0, "tar failed:\n" + r.stderr[-1500:]
            hits = glob.glob("/content/_extract/**/src/Router.cpp", recursive=True)
            assert hits, "Router.cpp not found inside archive"
            root = os.path.dirname(os.path.dirname(hits[0]))
            if os.path.exists(PROJ):
                shutil.move(PROJ, "/content/noxim_old_partial")
            shutil.move(root, PROJ)
            print(f"Extracted to {PROJ}")
    except Exception as e:
        print(f"Upload/Extraction skipped or encountered an error: {e}")

if os.path.exists(ROUTER):
    print("\n== key files ==")
    print(sh(f"ls -l {PROJ}/build.sh {PROJ}/bin/Makefile {PROJ}/app.py {PROJ}/*.pkl {PROJ}/node6_attack_comparison.csv 2>&1").stdout)
    print("Prebuilt binary present:", os.path.exists(f"{PROJ}/bin/noxim"))

    print("\n== bin/Makefile (full) ==")
    print(sh(f"cat {PROJ}/bin/Makefile").stdout)

    print("\n== build.sh ==")
    print(sh(f"cat {PROJ}/build.sh").stdout)
else:
    print("\n[Project Source Bypass] C++ source folder is not present on disk. Ready to train/analyze the uploaded simulation datasets.")

Please upload a Noxim source archive (.tar.gz/.tgz) if you wish to compile C++ source, or cancel if you only wish to analyze CSVs:


Saving attack_101.csv to attack_101 (2).csv
Saving attack_102.csv to attack_102 (2).csv
Saving attack_103.csv to attack_103 (2).csv
Saving attack_104.csv to attack_104 (2).csv
Saving attack_105.csv to attack_105 (2).csv
Saving attack_106.csv to attack_106 (2).csv
Saving attack_107.csv to attack_107 (2).csv
Saving attack_108.csv to attack_108 (2).csv
Saving attack_109.csv to attack_109 (2).csv
Saving attack_110.csv to attack_110 (2).csv
Saving attack_node27.csv to attack_node27 (2).csv
Saving blackhole_001.csv to blackhole_001 (2).csv
Saving blackhole_002.csv to blackhole_002 (2).csv
Saving BLACKHOLE_101.csv to BLACKHOLE_101 (2).csv
Saving BLACKHOLE_102.csv to BLACKHOLE_102 (2).csv
Saving BLACKHOLE_103.csv to BLACKHOLE_103 (2).csv
Saving BLACKHOLE_104.csv to BLACKHOLE_104 (2).csv
Saving BLACKHOLE_105.csv to BLACKHOLE_105 (2).csv
Saving blackhole_test.csv to blackhole_test (2).csv
Saving dos_flooding_001.csv to dos_flooding_001 (2).csv
Saving DOS_FLOODING_101.csv to DOS_FLOODING_101 (2).

In [ ]:
import os, subprocess, pandas as pd, json
from pathlib import Path

# Step 1: Upload CSVs to /content/noxim/results
os.makedirs("/content/noxim/results", exist_ok=True)
print("Upload all your CSV files now (the ones you just selected):")
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    target = f"/content/noxim/results/{fname}"
    with open(target, 'wb') as f:
        f.write(uploaded[fname])
print(f"\n✓ {len(uploaded)} CSVs saved to /content/noxim/results/")

# Step 2: List what we have
import glob
csvs = glob.glob("/content/noxim/results/*.csv")
print(f"Total CSVs: {len(csvs)}")
attack_types = set()
for csv in csvs:
    name = Path(csv).stem.upper()
    if any(x in name for x in ["BLACKHOLE", "ROUTING", "NORMAL", "DOS_FLOODING", "GRAYHOLE", "SELECTIVE", "TAMPERING"]):
        attack_types.add(name.split('_')[0])
print(f"Attack types detected: {sorted(attack_types)}")

# Step 3: Parse one CSV to show structure
sample = csvs[0]
df = pd.read_csv(sample)
print(f"\nSample CSV ({Path(sample).name}):")
print(df.head(2))
print("Columns:", df.columns.tolist())

Upload all your CSV files now (the ones you just selected):


Saving attack_101.csv to attack_101 (3).csv
Saving attack_102.csv to attack_102 (3).csv
Saving attack_103.csv to attack_103 (3).csv
Saving attack_104.csv to attack_104 (3).csv
Saving attack_105.csv to attack_105 (3).csv
Saving attack_106.csv to attack_106 (3).csv
Saving attack_107.csv to attack_107 (3).csv
Saving attack_108.csv to attack_108 (3).csv
Saving attack_109.csv to attack_109 (3).csv
Saving attack_110.csv to attack_110 (3).csv
Saving attack_node27.csv to attack_node27 (3).csv
Saving blackhole_001.csv to blackhole_001 (3).csv
Saving blackhole_002.csv to blackhole_002 (3).csv
Saving BLACKHOLE_101.csv to BLACKHOLE_101 (3).csv
Saving BLACKHOLE_102.csv to BLACKHOLE_102 (3).csv
Saving BLACKHOLE_103.csv to BLACKHOLE_103 (3).csv
Saving BLACKHOLE_104.csv to BLACKHOLE_104 (3).csv
Saving BLACKHOLE_105.csv to BLACKHOLE_105 (3).csv
Saving blackhole_test.csv to blackhole_test (3).csv
Saving dos_flooding_001.csv to dos_flooding_001 (3).csv
Saving DOS_FLOODING_101.csv to DOS_FLOODING_101 (3).

In [ ]:
import re
from pathlib import Path

import pandas as pd
import gradio as gr
import plotly.express as px

DATA_DIR = Path("/content/noxim/results")

if not DATA_DIR.exists():
    raise RuntimeError(
        "CSV folder missing. Re-run the previous CSV-upload cell."
    )

records = []

def classify(filename):
    name = filename.upper()
    # Check specific attack names before generic terms.
    patterns = [
        ("Selective forwarding", "SELECTIVE_FORWARDING"),
        ("Destination tampering", "DESTINATION_TAMPERING"),
        ("Routing manipulation", "ROUTING_MANIPULATION"),
        ("Blackhole", "BLACKHOLE"),
        ("Grayhole", "GRAYHOLE"),
        ("DoS flooding", "DOS_FLOODING"),
        ("DoS flooding", "DOS"),
        ("Normal baseline", "NORMAL"),
        ("Attack type unspecified", "ATTACK"),
    ]
    for label, key in patterns:
        if key in name:
            return label
    return "Unknown scenario"

for path in sorted(DATA_DIR.glob("*.csv")):
    try:
        raw = pd.read_csv(path)
        if raw.empty:
            continue

        row = raw.iloc[0]
        stem = re.sub(r"\s*\(\d+\)$", "", path.stem)

        ratio = row.get("received_ideal_flits_ratio")
        ratio = pd.to_numeric(pd.Series([ratio]), errors="coerce").iloc[0]

        delay = pd.to_numeric(
            pd.Series([row.get("global_average_delay_cycles")]),
            errors="coerce"
        ).iloc[0]

        throughput = pd.to_numeric(
            pd.Series([row.get("network_throughput_flits_per_cycle")]),
            errors="coerce"
        ).iloc[0]

        node_match = re.search(r"NODE[_-]?(\d+)", stem, re.I)

        records.append({
            "File": path.name,
            "Scenario": classify(stem),
            "Node in filename": (
                int(node_match.group(1)) if node_match else None
            ),
            "Topology": (
                f"{int(row['mesh_dim_x'])}×{int(row['mesh_dim_y'])}"
                if pd.notna(row.get("mesh_dim_x"))
                and pd.notna(row.get("mesh_dim_y"))
                else "Unknown"
            ),
            "Seed": row.get("rnd_generator_seed"),
            "Received packets": row.get("total_received_packets"),
            "Received flits": row.get("total_received_flits"),
            "Ratio metric (%)": ratio * 100 if pd.notna(ratio) else None,
            "Average delay (cycles)": delay,
            "Throughput": throughput,
            "Ratio warning": (
                "Investigate ratio >100%"
                if pd.notna(ratio) and ratio > 1
                else "Within 100%"
                if pd.notna(ratio) else "Metric unavailable"
            ),
        })
    except Exception as e:
        print(f"⚠️ Skipping corrupted file {path.name}: {e}")

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError("No valid one-row Noxim CSVs were found.")

print(f"Loaded {len(df)} simulation CSVs.")
print("No C++ build or ML model is needed for this dashboard.")

metrics = {
    "Received/ideal-flits ratio (%)": "Ratio metric (%)",
    "Average delay (cycles)": "Average delay (cycles)",
    "Network throughput": "Throughput",
}

def render(scenario, metric):
    view = df if scenario == "All scenarios" else df[df["Scenario"] == scenario]

    if view.empty:
        return "No records for this selection.", None, view

    summary = (
        f"**Runs shown:** {len(view)}  \n"
        f"**Scenario groups:** {view['Scenario'].nunique()}  \n"
        f"**Ratios above 100%:** "
        f"{(pd.to_numeric(view['Ratio metric (%)'], errors='coerce') > 100).sum()}"
    )

    field = metrics[metric]
    plotted = (
        view.groupby("Scenario", as_index=False)
        .agg(Value=(field, "median"), Runs=("File", "count"))
    )

    fig = px.bar(
        plotted,
        x="Scenario",
        y="Value",
        text_auto=".3f",
        hover_data=["Runs"],
        title=f"Median {metric} by scenario",
        template="plotly_dark",
        color="Scenario",
    )
    fig.update_layout(
        showlegend=False,
        height=430,
        margin=dict(l=25, r=25, t=65, b=100),
        xaxis_title="Scenario inferred from filename",
        yaxis_title=metric,
    )

    return summary, fig, view

css = """
.gradio-container {
    background: linear-gradient(140deg, #0b1220, #14243b);
}
"""

with gr.Blocks(title="NoC Sentinel | Results Intelligence", css=css) as demo:
    gr.Markdown("""
    # 🛡️ NoC Sentinel
    ### Network-on-Chip · Simulation Results Intelligence

    Explore your saved Noxim experiments without rebuilding the simulator.
    """)
    with gr.Row():
        scenario = gr.Dropdown(
            ["All scenarios"] + sorted(df["Scenario"].unique().tolist()),
            value="All scenarios",
            label="Scenario filter",
        )
        metric = gr.Dropdown(
            list(metrics.keys()),
            value="Received/ideal-flits ratio (%)",
            label="Metric to visualize",
        )

    summary = gr.Markdown()
    plot = gr.Plot()
    table = gr.Dataframe(
        value=df,
        label="Simulation records",
        interactive=False,
    )

    scenario.change(
        render, inputs=[scenario, metric],
        outputs=[summary, plot, table]
    )
    metric.change(
        render, inputs=[scenario, metric],
        outputs=[summary, plot, table]
    )
    demo.load(
        render, inputs=[scenario, metric],
        outputs=[summary, plot, table]
    )

demo.launch(share=True)

Loaded 50 simulation CSVs.
No C++ build or ML model is needed for this dashboard.


/tmp/ipykernel_515/671132137.py:147: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="NoC Sentinel | Results Intelligence", css=css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d525e0cdd52246de6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

!pip -q install gradio pandas plotly

import html
import pandas as pd
import gradio as gr
import plotly.graph_objects as go

COLUMNS = [
    "attack", "target_node", "before_impact_pct",
    "after_impact_pct", "backup_node", "route_verified"
]

def run_dashboard(file):
    if file is None:
        return "<p>Upload one consolidated CSV.</p>", pd.DataFrame(), go.Figure(), "Waiting for file"

    try:
        df = pd.read_csv(file)
        missing = [c for c in COLUMNS if c not in df.columns]
        if missing:
            return (
                "<p>Missing columns: " + html.escape(", ".join(missing)) + "</p>",
                pd.DataFrame(columns=COLUMNS),
                go.Figure(),
                "Please use the required CSV format."
            )

        for c in ["before_impact_pct", "after_impact_pct"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df["impact_reduction_pp"] = (
            df["before_impact_pct"] - df["after_impact_pct"]
        )
        df["route_verified"] = (
            df["route_verified"].astype(str).str.lower()
            .isin(["true", "yes", "1", "verified"])
        )

        target = html.escape(str(df.iloc[0]["target_node"]))
        backup = html.escape(str(df.iloc[0]["backup_node"]))
        verified = bool(df["route_verified"].all())
        color = "#22c55e" if verified else "#f59e0b"
        label = "ROUTE VERIFIED" if verified else "ROUTE NOT VERIFIED"

        # Three communication clusters inspired by your reference image.
        topology = f"""
        <div style="background:#08111f;color:white;padding:16px;border-radius:16px">
        <h3>NoC Sentinel · Network Intelligence</h3>
        <p style="color:{color}"><b>{label}</b></p>
        <svg viewBox="0 0 600 300" width="100%">
          <ellipse cx="110" cy="150" rx="90" ry="115"
                   fill="none" stroke="#64748b" stroke-dasharray="4 5"/>
          <ellipse cx="470" cy="90" rx="105" ry="75"
                   fill="none" stroke="#64748b" stroke-dasharray="4 5"/>
          <ellipse cx="340" cy="245" rx="95" ry="45"
                   fill="none" stroke="#64748b" stroke-dasharray="4 5"/>
          <g stroke="#718096" stroke-width="2">
            <path d="M110 150 L55 95 M110 150 L90 65 M110 150 L165 105
                     M110 150 L55 200 M110 150 L160 205"/>
            <path d="M390 90 L360 45 M390 90 L450 40 M390 90 L500 70
                     M390 90 L475 125 M390 90 L420 140"/>
            <path d="M340 245 L295 225 M340 245 L365 275 M340 245 L395 230"/>
            <path d="M110 150 L390 90 M390 90 L340 245"/>
          </g>
          <path d="M110 150 Q230 190 340 245 Q370 175 390 90"
                fill="none" stroke="{color}" stroke-width="4"
                stroke-dasharray="8 7">
            <animate attributeName="stroke-dashoffset"
                     from="30" to="0" dur="1.2s" repeatCount="indefinite"/>
          </path>
          <g fill="#06b6d4" stroke="#dbeafe">
            <circle cx="55" cy="95" r="8"/><circle cx="90" cy="65" r="8"/>
            <circle cx="165" cy="105" r="8"/><circle cx="55" cy="200" r="8"/>
            <circle cx="160" cy="205" r="8"/><circle cx="360" cy="45" r="8"/>
            <circle cx="450" cy="40" r="8"/><circle cx="500" cy="70" r="8"/>
            <circle cx="475" cy="125" r="8"/><circle cx="420" cy="140" r="8"/>
            <circle cx="295" cy="225" r="8"/><circle cx="365" cy="275" r="8"/>
            <circle cx="395" cy="230" r="8"/>
          </g>
          <g fill="#ef4444" stroke="white" stroke-width="2">
            <circle cx="110" cy="150" r="12"/>
            <circle cx="390" cy="90" r="12"/>
            <circle cx="340" cy="245" r="12"/>
          </g>
          <g fill="white" font-size="12" font-weight="bold">
            <text x="25" y="175">Node {target} · Target</text>
            <text x="405" y="110">Backup {backup}</text>
            <text x="295" y="220">Core</text>
          </g>
        </svg>
        <small style="color:#9ca3af">
        Animated illustrative topology; the route is not a measured packet trace.
        </small>
        </div>
        """

        fig = go.Figure()
        fig.add_bar(
            x=df["attack"], y=df["before_impact_pct"],
            name="Before mitigation"
        )
        fig.add_bar(
            x=df["attack"], y=df["after_impact_pct"],
            name="After mitigation"
        )
        fig.update_layout(
            barmode="group",
            title="Attack impact: before vs after",
            yaxis_title="Impact (%)",
            template="plotly_dark"
        )

        status = (
            f"Loaded {len(df)} scenarios. "
            "Positive reduction means lower reported impact. "
            "Route status depends on the uploaded evidence."
        )
        return topology, df, fig, status

    except Exception as e:
        return "<p>CSV error: " + html.escape(str(e)) + "</p>", pd.DataFrame(), go.Figure(), "Could not read file"

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="cyan"),
    title="NoC Sentinel"
) as app:
    gr.Markdown("# 🛡️ NoC Sentinel\nUpload **one CSV containing all five attack scenarios**.")
    file = gr.File(label="Upload consolidated results CSV", file_types=[".csv"], type="filepath")
    status = gr.Markdown("Waiting for upload.")
    topology = gr.HTML()
    chart = gr.Plot(label="Mitigation comparison")
    table = gr.Dataframe(label="Scenario results", interactive=False)
    file.change(
        run_dashboard, inputs=file,
        outputs=[topology, table, chart, status]
    )

app.launch(share=True)


/tmp/ipykernel_4492/831138000.py:123: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65d8b3433002f14b46.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# NoC Sentinel — Attack Intelligence + Defense & Recovery
# Google Colab: run this cell once, then upload a comparison report and/or up to 25 CSV/log files.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio', 'pandas', 'plotly'], check=True)

import os, re, json, zipfile, tempfile, urllib.request, urllib.error, html
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import gradio as gr

MODEL_NAME = "gemini-3.8-flash"
MAX_FILES = 25

CSS = r"""
:root { color-scheme: dark; }
body, .gradio-container { background:#07111f !important; color:#e5eefb !important; }
.gradio-container { width:100% !important; max-width:1700px !important; margin:0 auto !important; }
main, .main, .wrap { background:#07111f !important; color:#e5eefb !important; }
h1,h2,h3,p,label,span,div { }
.gradio-container, .gradio-container * { --body-text-color:#e5eefb; --block-label-text-color:#a8bad0; }
.hero {background:linear-gradient(125deg,#101f35 0%,#0a1525 60%,#0b2932 100%);border:1px solid #213953;border-radius:22px;padding:28px 32px;margin-bottom:14px}
.eyebrow {font-size:11px;letter-spacing:2.2px;text-transform:uppercase;color:#44e1d1;font-weight:800}
.hero h1 {font-size:34px;line-height:1.15;margin:8px 0;color:#f3f7ff}
.hero p {color:#9fb3cc;margin:0;font-size:15px}
.section-title {font-size:20px;font-weight:800;color:#eff6ff;margin:8px 0}
.subtle {color:#9fb3c8;font-size:13px}
.kpi-grid {display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:10px 0 18px}
.kpi {background:#0e1b2d;border:1px solid #21364f;border-radius:16px;padding:17px;min-height:91px}
.kpi .label {font-size:10px;letter-spacing:1.5px;text-transform:uppercase;color:#8ea6c2;font-weight:700}
.kpi .value {font-size:27px;line-height:1.4;font-weight:800;color:#f3f7ff}
.kpi .hint {font-size:11px;color:#8ea6c2}
.panel {background:#0b1829;border:1px solid #203650;border-radius:18px;padding:16px}
.status-warn {color:#fbbf24;font-weight:800}
.status-ok {color:#4ade80;font-weight:800}
.status-bad {color:#fb7185;font-weight:800}
footer {display:none !important}

.topology-wrap{background:#0b1829;border:1px solid #213851;border-radius:20px;padding:18px;margin:14px 0;color:#e5eefb}
.topology-head{display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap}
.route-badge{border:1px solid;border-radius:999px;padding:7px 11px;font-size:10px;letter-spacing:1px;font-weight:900}
.zone{fill:#10243b;stroke:#45617d;stroke-dasharray:5 5;stroke-width:1.4}
.edge{stroke:#5b718d;stroke-width:1.4;opacity:.85}
.peer{fill:#12bde4;stroke:#05101d;stroke-width:2}
.target{fill:#fb4f69;stroke:#fff;stroke-width:2}
.backup-ring{fill:none;stroke:#fbbf24;stroke-width:5}
.core{fill:#12bde4;stroke:#e0f2fe;stroke-width:2}
.animated-route{fill:none;stroke-width:4;stroke-dasharray:9 8;animation:routeFlow 1.2s linear infinite}
@keyframes routeFlow{to{stroke-dashoffset:-34}}
.zonelabel{fill:#a9bfd8;font-size:10px;letter-spacing:1.1px;font-weight:800}
.node-label{fill:#f2f7ff;font-size:11px;font-weight:800;paint-order:stroke;stroke:#0b1829;stroke-width:3px}
.topology-legend{display:flex;gap:18px;flex-wrap:wrap;margin-top:8px;font-size:11px;color:#a9bfd8}
.dot{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px}
.red{background:#fb4f69}.blue{background:#12bde4}.amber{background:#fbbf24}.green{background:#4ade80}

@media(max-width:800px){.kpi-grid{grid-template-columns:repeat(2,minmax(0,1fr))}.hero{padding:20px}.hero h1{font-size:26px}}
"""

def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

def colmap(df):
    return {norm(c): c for c in df.columns}

def find_col(df, candidates):
    mapping = colmap(df)
    for cand in candidates:
        if norm(cand) in mapping:
            return mapping[norm(cand)]
    # conservative partial match
    for cand in candidates:
        n = norm(cand)
        for key, original in mapping.items():
            if n and (n in key or key in n):
                return original
    return None

def number(value):
    try:
        x = float(str(value).replace("%","").replace(",","").strip())
        return x if pd.notna(x) else None
    except Exception:
        return None

def percent_metric(value, source_col=""):
    x = number(value)
    if x is None:
        return None
    # Noxim CSVs often store the ratio in [0, ~1+] while comparison
    # reports may already store percent values.
    if "ratio" in norm(source_col) and -2 <= x <= 2:
        return x * 100
    return x

ATTACK_TEXT = {
    "blackhole": ("Blackhole", "A malicious router discards transit traffic. Look for packet-drop logs and lower delivered traffic; delay alone does not prove the attack."),
    "selectiveforwarding": ("Selective Forwarding", "The node drops selected packets while forwarding others. A credible finding needs packet/drop evidence, not only a scenario filename."),
    "grayhole": ("Grayhole", "The node intermittently drops traffic. Compare controlled runs and drop evidence because short runs may hide intermittent behavior."),
    "routingmanipulation": ("Routing Manipulation", "The node alters route choice. Verify the next-hop/path trace; a route-hook message alone does not prove successful bypass."),
    "destinationtampering": ("Destination Tampering", "Destination metadata is changed. Verify source/destination fields and whether traffic reached the intended or failover endpoint."),
    "dosflooding": ("DoS Flooding", "The scenario suggests excessive traffic. Check injection rate, congestion, packet loss and latency against a matched baseline."),
    "baseline": ("Baseline", "Reference run without the named attack. Compare only runs using equivalent simulation settings."),
    "normal": ("Normal Traffic", "Reference/normal sample; its label should be confirmed from the test setup.")
}

def infer_attack(text):
    n = norm(text)
    # Order prevents more generic keys from taking a specific match.
    for key in ["destinationtampering", "selectiveforwarding", "routingmanipulation",
                "blackhole", "grayhole", "dosflooding", "baseline", "normal"]:
        if key in n:
            return ATTACK_TEXT[key][0], ATTACK_TEXT[key][1], True
    m = re.search(r"attack[_ -]?(\d+)", str(text), re.I)
    if m:
        return f"Attack sample {m.group(1)} (type unspecified)", "The filename labels this as an attack sample, but it does not identify the attack type. Add a matching Noxim log or descriptive filename to attribute the attack.", False
    return Path(str(text)).stem.replace("_"," ").title() or "Unlabelled sample", "Attack type not identified from the supplied filename or report. Do not infer a specific attack without scenario/log evidence.", False

def truthy(v):
    return str(v).strip().lower() in {"1","true","yes","observed","verified","confirmed","pass","passed"}

def read_text_file(path):
    try:
        return Path(path).read_text(encoding="utf-8", errors="ignore")[:250000]
    except Exception:
        return ""

def expand_uploads(files):
    paths = []
    tempdirs = []
    if not files:
        return paths, tempdirs, []
    if not isinstance(files, list):
        files = [files]
    capped = files[:MAX_FILES]
    notices = []
    if len(files) > MAX_FILES:
        notices.append(f"Received {len(files)} files; processing the first {MAX_FILES} only.")
    for f in capped:
        p = Path(f)
        if p.suffix.lower() == ".zip":
            td = tempfile.TemporaryDirectory()
            tempdirs.append(td)
            with zipfile.ZipFile(p) as z:
                for info in z.infolist():
                    # Prevent ZIP path traversal, ignore hidden/unsupported files.
                    name = Path(info.filename)
                    if name.is_absolute() or ".." in name.parts:
                        continue
                    if name.suffix.lower() not in {".csv", ".log", ".txt"}:
                        continue
                    dest = Path(td.name) / name.name
                    dest.write_bytes(z.read(info))
                    paths.append(str(dest))
        elif p.suffix.lower() in {".csv", ".log", ".txt"}:
            paths.append(str(p))
    if len(paths) > MAX_FILES:
        notices.append(f"Processing capped at {MAX_FILES} evidence files.")
        paths = paths[:MAX_FILES]
    return paths, tempdirs, notices

def bool_from_row(row, candidates):
    for cand in candidates:
        c = find_col(pd.DataFrame([row]), [cand])
        if c:
            return truthy(row.get(c))
    return False

def safe_val(row, candidates):
    # row has raw columns, use normalized matching
    keys = {norm(k): k for k in row.keys()}
    for cand in candidates:
        if norm(cand) in keys:
            return row.get(keys[norm(cand)])
    for cand in candidates:
        cn = norm(cand)
        for k, orig in keys.items():
            if cn and (cn in k or k in cn):
                return row.get(orig)
    return None

def get_metric_value(df, candidates, percent=False):
    col = find_col(df, candidates)
    if not col or df.empty:
        return None, None
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    if vals.empty:
        return None, col
    # A simulator stats CSV may include one summary row or many samples.
    v = float(vals.mean())
    if percent:
        v = percent_metric(v, col)
    return v, col

def parse_comparison_csv(df, filename):
    attack_col = next((c for c in df.columns if norm(c) in {norm("attack"), norm("attack type"), norm("scenario"), norm("attack name")}), None)
    base_col = find_col(df, ["baseline ideal-flit metric (%)", "baseline metric (%)",
                             "baseline ratio", "baseline ideal flit ratio", "baseline"])
    attack_metric_col = find_col(df, ["under-attack metric (%)", "attack metric (%)",
                                      "attack ratio", "under attack ratio", "attack value"])
    mitigated_col = find_col(df, ["mitigated metric (%)", "mitigation metric (%)",
                                  "mitigated ratio", "after mitigation metric", "after metric"])
    # Only treat as a per-attack comparison if the required structure exists.
    if not (attack_col and (attack_metric_col or mitigated_col or base_col)):
        return []
    records = []
    for _, r in df.iterrows():
        name = str(r.get(attack_col, filename))
        label, explanation, identified = infer_attack(name)
        node = safe_val(r, ["target node", "affected node", "malicious node", "node id", "node"])
        base = percent_metric(r.get(base_col), base_col or "") if base_col else None
        attacked = percent_metric(r.get(attack_metric_col), attack_metric_col or "") if attack_metric_col else None
        mitigated = percent_metric(r.get(mitigated_col), mitigated_col or "") if mitigated_col else None
        records.append({
            "Attack": label if identified else name,
            "Scenario label": name,
            "Source file": filename,
            "Target node": node,
            "Baseline metric (%)": base,
            "Under-attack metric (%)": attacked,
            "Mitigated metric (%)": mitigated,
            "Attack delta vs baseline (pp)": number(safe_val(r, ["attack change vs baseline (pp)", "attack delta pp"])),
            "Mitigation change (pp)": number(safe_val(r, ["mitigation change (pp)", "mitigated change pp"])),
            "Attack delay (cycles)": number(safe_val(r, ["attack delay (cycles)", "under attack delay", "attack delay"])),
            "Mitigated delay (cycles)": number(safe_val(r, ["mitigated delay (cycles)", "mitigation delay", "mitigated delay"])),
            "Attack log evidence": truthy(safe_val(r, ["attack log evidence", "attack evidence", "attack observed"])),
            "Mitigation hook observed": truthy(safe_val(r, ["mitigation route log observed", "mitigation hook observed", "mitigation evidence"])),
            "Path verified": truthy(safe_val(r, ["path verified", "route verified", "verified path"])),
            "Destination delivery verified": truthy(safe_val(r, ["destination delivery verified", "delivery verified", "failover delivery verified"])),
            "Explanation": explanation
        })
    return records

def summarize_upload(files):
    paths, tempdirs, notices = expand_uploads(files)
    all_logs = {}
    csv_frames = []
    comparisons = []
    sample_rows = []
    evidence_rows = []
    detected_nodes = set()
    for path in paths:
        p = Path(path)
        stem = p.stem
        att_name, explain, identified = infer_attack(stem)
        node_match = re.search(r"(?:node|mal|target)[_-]?(\d+)", stem, re.I)
        file_node = int(node_match.group(1)) if node_match else None

        if p.suffix.lower() in {".log", ".txt"}:
            txt = read_text_file(path)
            all_logs[stem.lower()] = txt
            hooks = len(re.findall(r"MITIGATION:|MITIGATION_HOOK|mitigation hook", txt, re.I))
            attack_markers = re.findall(
                r"BLACKHOLE|SELECTIVE_FORWARDING|GRAYHOLE|ROUTING_MANIPULATION|DESTINATION_TAMPERING|DOS_FLOODING",
                txt, re.I
            )
            node_ids = set(int(x) for x in re.findall(r"(?:node|Node)\s*[_#]?\s*(\d+)", txt))
            detected_nodes.update(node_ids)
            path_marked = bool(re.search(r"PATH_VERIFIED|ROUTE_VERIFIED|VERIFIED_PATH", txt, re.I))
            delivery_marked = bool(re.search(r"DESTINATION_DELIVERY_VERIFIED|FAILOVER_DELIVERY_VERIFIED|DELIVERY_VERIFIED", txt, re.I))
            evidence_rows.append({
                "File": p.name, "Type": "Log / text",
                "Attack markers": ", ".join(sorted(set(s.upper() for s in attack_markers))) or "None found",
                "Node IDs mentioned": ", ".join(map(str, sorted(node_ids))) or "Not parsed",
                "Mitigation hook lines": hooks,
                "Path verified marker": path_marked,
                "Destination delivery marker": delivery_marked
            })
            continue

        try:
            df = pd.read_csv(path)
        except Exception as e:
            notices.append(f"Could not read {p.name}: {e}")
            continue
        if df.empty:
            notices.append(f"{p.name} is empty.")
            continue
        csv_frames.append((p.name, df))
        # Consolidated comparison file
        comp = parse_comparison_csv(df, p.name)
        if comp:
            comparisons.extend(comp)
            continue

        # Also understand common raw Noxim stats CSVs.
        attack_label, explanation, identified = infer_attack(p.name)
        node = file_node
        cnode = find_col(df, ["target node", "affected node", "malicious node", "node id"])
        if cnode and not df[cnode].empty:
            try:
                node = int(float(df[cnode].iloc[0]))
            except Exception:
                pass
        if node is not None:
            detected_nodes.add(node)
        ratio, ratio_col = get_metric_value(df, [
            "received ideal flits ratio", "received ideal flit ratio",
            "received_ideal_flits_ratio", "ideal flit ratio", "delivery ratio"
        ], percent=True)
        delay, delay_col = get_metric_value(df, [
            "global average delay (cycles)", "global average delay", "average delay",
            "global_avg_delay", "avg_delay", "average packet delay"
        ])
        throughput, through_col = get_metric_value(df, [
            "global throughput", "throughput", "average throughput"
        ])
        scenario_col = find_col(df, ["attack", "scenario", "attack type", "mode"])
        if scenario_col:
            scenario_text = str(df[scenario_col].iloc[0])
            attack_label, explanation, identified = infer_attack(scenario_text)
        sample_rows.append({
            "Attack / scenario": attack_label,
            "Type identified": identified,
            "Source file": p.name,
            "Target node": node,
            "Ideal-flit metric (%)": ratio,
            "Average delay (cycles)": delay,
            "Throughput": throughput,
            "Rows": len(df),
            "Evidence basis": "Filename / scenario column; not a new ML prediction",
            "Explanation": explanation
        })

    # Parse attack/route markers from all logs and link explicit named evidence where possible.
    combined_logs = "\n".join(all_logs.values())
    for rec in comparisons:
        # Associate logs by scenario-specific filename/content only; do not let one generic log
        # promote every attack row to verified.
        scen_key = norm(rec.get("Scenario label", "")) or norm(rec.get("Attack", ""))
        relevant_logs = [
            txt for log_name, txt in all_logs.items()
            if (scen_key and (scen_key in norm(log_name) or scen_key in norm(txt)))
        ]
        joined_relevant = "\\n".join(relevant_logs)
        if not rec.get("Attack log evidence") and relevant_logs:
            rec["Attack log evidence"] = bool(re.search(
                r"BLACKHOLE|SELECTIVE_FORWARDING|GRAYHOLE|ROUTING_MANIPULATION|DESTINATION_TAMPERING|DOS_FLOODING|drop|attack",
                joined_relevant, re.I
            ))
        # A hook message does not prove a path or endpoint delivery.
        if not rec.get("Path verified"):
            rec["Path verified"] = bool(re.search(r"PATH_VERIFIED|ROUTE_VERIFIED|VERIFIED_PATH", joined_relevant, re.I))
        if not rec.get("Destination delivery verified"):
            rec["Destination delivery verified"] = bool(re.search(r"DESTINATION_DELIVERY_VERIFIED|FAILOVER_DELIVERY_VERIFIED|DELIVERY_VERIFIED", joined_relevant, re.I))

    all_scenarios = set()
    for r in comparisons:
        all_scenarios.add(r["Attack"])
    for r in sample_rows:
        all_scenarios.add(r["Attack / scenario"])

    # Prefer explicit node IDs from comparison data, then source/log evidence.
    for r in comparisons:
        try:
            detected_nodes.add(int(float(r["Target node"])))
        except Exception:
            pass
    target_node = 6 if 6 in detected_nodes else (min(detected_nodes) if detected_nodes else None)

    report = {
        "files": paths, "tempdirs": tempdirs, "notices": notices,
        "comparisons": comparisons, "samples": sample_rows, "evidence": evidence_rows,
        "target_node": target_node, "scenario_count": len(all_scenarios),
        "comparison_df": pd.DataFrame(comparisons),
        "sample_df": pd.DataFrame(sample_rows),
        "evidence_df": pd.DataFrame(evidence_rows),
        "raw_logs": all_logs
    }
    return report

def metric_chart(report, defense=False):
    fig = go.Figure()
    comps = report["comparisons"]
    samples = report["samples"]
    if comps:
        df = pd.DataFrame(comps)
        x = df["Attack"]
        for col, name in [
            ("Baseline metric (%)", "Baseline"),
            ("Under-attack metric (%)", "Under attack"),
            ("Mitigated metric (%)", "Mitigation enabled")
        ]:
            if col in df and df[col].notna().any():
                fig.add_bar(x=x, y=df[col], name=name)
        fig.update_layout(barmode="group", title="Measured Noxim metric · baseline / attack / mitigation",
                          yaxis_title="Ideal-flit ratio metric (%) — not attack probability")
        # Deliberately don't clamp; Noxim ratio may exceed 100.
    elif samples:
        df = pd.DataFrame(samples)
        if "Ideal-flit metric (%)" in df and df["Ideal-flit metric (%)"].notna().any():
            fig.add_bar(x=df["Attack / scenario"], y=df["Ideal-flit metric (%)"], name="Ideal-flit metric")
            fig.update_layout(title="Available Noxim measurements by uploaded scenario",
                              yaxis_title="Ideal-flit ratio metric (%) — not attack probability")
        elif "Average delay (cycles)" in df and df["Average delay (cycles)"].notna().any():
            fig.add_bar(x=df["Attack / scenario"], y=df["Average delay (cycles)"], name="Average delay")
            fig.update_layout(title="Average delay by uploaded scenario", yaxis_title="Cycles")
        else:
            fig.add_annotation(text="No recognised numeric Noxim metric found in uploaded CSVs",
                               x=.5,y=.5,xref="paper",yref="paper",showarrow=False)
            fig.update_layout(title="Simulation measurements")
    else:
        fig.add_annotation(text="Upload Noxim CSVs or a comparison report to populate the chart",
                           x=.5,y=.5,xref="paper",yref="paper",showarrow=False)
        fig.update_layout(title="Waiting for simulation evidence")
    fig.update_layout(
        template="plotly_dark", paper_bgcolor="#0b1829", plot_bgcolor="#0b1829",
        font=dict(color="#e5eefb"), legend=dict(orientation="h",y=1.12),
        margin=dict(l=20,r=20,t=76,b=90), height=390
    )
    fig.update_xaxes(tickangle=-18, gridcolor="#1e334a")
    fig.update_yaxes(gridcolor="#1e334a", rangemode="tozero")
    return fig

def svg_topology(target=6, backup=7, path_verified=False, delivered=False):
    # Clustered topology inspired by the uploaded reference. Animated route remains marked illustrative
    # unless uploaded telemetry explicitly verifies a path. This is not a fabricated measured trace.
    target = html.escape(str(target if target is not None else "Unknown"))
    backup = html.escape(str(backup))
    color = "#4ade80" if path_verified else "#fbbf24"
    status = "PATH VERIFIED" if path_verified else "ILLUSTRATIVE PATH · NOT VERIFIED"
    backup_status = "DELIVERY VERIFIED" if delivered else "BACKUP CANDIDATE"
    # coordinates
    left_nodes = [(96,135),(140,76),(203,117),(211,179),(100,207),(66,258),(154,245),(220,224)]
    right_nodes = [(408,83),(458,53),(520,79),(574,111),(542,165),(484,171),(445,146)]
    bottom_nodes = [(300,253),(340,276),(389,253),(417,290),(348,310)]
    edges=[]
    for x,y in left_nodes: edges.append((165,175,x,y))
    for x,y in right_nodes: edges.append((468,112,x,y))
    for x,y in bottom_nodes: edges.append((327,267,x,y))
    edges += [(165,175,468,112),(468,112,327,267)]
    edge_svg="".join(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" class="edge"/>' for x1,y1,x2,y2 in edges)
    nodes_svg=""
    for x,y in left_nodes+right_nodes+bottom_nodes:
        nodes_svg += f'<circle cx="{x}" cy="{y}" r="8" class="peer"/>'
    return f"""
    <div class="topology-wrap">
      <div class="topology-head">
        <div><div class="section-title">Communication fabric</div>
        <div class="subtle">Clustered network reference · target node {target} · backup node {backup}</div></div>
        <div class="route-badge" style="border-color:{color};color:{color}">{status}</div>
      </div>
      <svg viewBox="0 0 650 340" role="img" aria-label="Clustered network diagram">
        <defs><marker id="routearrow" markerWidth="8" markerHeight="8" refX="5" refY="3" orient="auto"><path d="M0,0 L6,3 L0,6 Z" fill="{color}"/></marker></defs>
        <ellipse cx="155" cy="170" rx="128" ry="133" class="zone"/>
        <ellipse cx="485" cy="112" rx="143" ry="82" class="zone"/>
        <ellipse cx="360" cy="280" rx="105" ry="48" class="zone"/>
        {edge_svg}
        <path d="M165 175 Q288 208 327 267 Q385 191 468 112" class="animated-route" style="stroke:{color}" marker-end="url(#routearrow)"/>
        {nodes_svg}
        <circle cx="165" cy="175" r="13" class="target"/>
        <circle cx="468" cy="112" r="16" class="backup-ring"/>
        <circle cx="468" cy="112" r="8" class="peer"/>
        <circle cx="327" cy="267" r="11" class="core"/>
        <text x="77" y="32" class="zonelabel">COMMUNICATION AREA A</text>
        <text x="407" y="19" class="zonelabel">COMMUNICATION AREA B</text>
        <text x="306" y="333" class="zonelabel">COMMUNICATION AREA C</text>
        <text x="88" y="201" class="node-label">Compromised {target}</text>
        <text x="489" y="142" class="node-label">Backup {backup}</text>
        <text x="344" y="262" class="node-label">Core</text>
      </svg>
      <div class="topology-legend">
        <span><i class="dot red"></i>Compromised target</span>
        <span><i class="dot blue"></i>Neighbor nodes</span>
        <span><i class="dot amber"></i>Backup candidate</span>
        <span><i class="dot green"></i>Verified route only</span>
      </div>
      <div class="subtle" style="margin-top:8px">Backup role: {backup_status}. Moving packets are a UI animation; this SVG becomes a verified route only when the uploaded simulator evidence explicitly confirms the path and delivery.</div>
    </div>
    """

def html_kpis(items):
    cards = []
    for label, value, hint in items:
        cards.append(f'<div class="kpi"><div class="label">{html.escape(str(label))}</div><div class="value">{html.escape(str(value))}</div><div class="hint">{html.escape(str(hint))}</div></div>')
    return '<div class="kpi-grid">' + ''.join(cards) + '</div>'

def analyze_files(files):
    if not files:
        msg = html_kpis([
            ("Files analysed","0","Upload 1–25 CSV / log files"),
            ("Attack scenarios","0","Inferred from report / labels"),
            ("Affected node","—","Node ID from evidence"),
            ("Verified routes","0","Requires explicit route proof")
        ])
        empty = go.Figure()
        empty.add_annotation(text="Upload simulation evidence to populate this chart", x=.5,y=.5,xref="paper",yref="paper",showarrow=False)
        empty.update_layout(template="plotly_dark",paper_bgcolor="#0b1829",plot_bgcolor="#0b1829")
        return msg, "<div class='panel'>Upload your comparison report and/or multiple scenario CSV/log files.</div>", pd.DataFrame(), empty, msg, svg_topology(), empty, pd.DataFrame(), ""
    report = summarize_upload(files)
    comps = report["comparisons"]
    samples = report["samples"]
    evidence = report["evidence"]
    target = report["target_node"]
    paths = report["files"]
    n_hooks = sum(int(r.get("Mitigation hook observed", False)) for r in comps)
    n_verified = sum(int(r.get("Path verified", False)) for r in comps)
    n_delivery = sum(int(r.get("Destination delivery verified", False)) for r in comps)
    n_evidence = sum(bool(r.get("Attack log evidence", False)) for r in comps)
    if not comps:
        n_hooks = sum(r.get("Mitigation hook lines", 0) > 0 for r in evidence)
        n_verified = sum(bool(r.get("Path verified marker")) for r in evidence)
        n_delivery = sum(bool(r.get("Destination delivery marker")) for r in evidence)
        n_evidence = sum(bool(r.get("Attack markers") not in ["None found"]) for r in evidence)
    scenario_labels = sorted(set([r["Attack"] for r in comps] + [r["Attack / scenario"] for r in samples]))
    intel_kpis = html_kpis([
        ("Files analysed", f"{len(paths)}", f"Maximum {MAX_FILES} files per analysis"),
        ("Scenario labels", str(len(scenario_labels)), "From report rows / filenames"),
        ("Affected node", str(target if target is not None else "Not found"), "Only when supported by data"),
        ("Attack evidence", f"{n_evidence}", "Rows with attack-log markers")
    ])
    def_kpis = html_kpis([
        ("Mitigation hooks", str(n_hooks), "Hook observed ≠ success"),
        ("Path verified", f"{n_verified}", "Needs explicit route evidence"),
        ("Delivery verified", f"{n_delivery}", "Needs explicit failover delivery"),
        ("Candidate backup", "7", "Set by demo assumption unless report specifies it")
    ])
    explanations = []
    for r in comps:
        baseline, attacked, mitigated = r.get("Baseline metric (%)"), r.get("Under-attack metric (%)"), r.get("Mitigated metric (%)")
        detail = r.get("Explanation", "")
        if baseline is not None and attacked is not None:
            delta = attacked - baseline
            detail += f" Reported attack metric differs from baseline by {delta:+.2f} percentage points."
        if attacked is not None and mitigated is not None:
            delta2 = attacked - mitigated
            detail += f" Reported metric change from attack to mitigation is {delta2:+.2f} points (direction depends on what this metric means)."
        detail += " A route-hook log is not proof of end-to-end delivery."
        explanations.append(detail)
    comp_df = pd.DataFrame(comps)
    sample_df = pd.DataFrame(samples)
    if not comp_df.empty:
        table = comp_df
    elif not sample_df.empty:
        table = sample_df
    else:
        table = pd.DataFrame(columns=["Status"])
    ev_df = pd.DataFrame(evidence)
    if comp_df.empty and ev_df.empty:
        ev_df = pd.DataFrame([{"Evidence":"No log/text evidence uploaded","Status":"Not verified"}])
    if report["notices"]:
        warning = "<br>".join(html.escape(x) for x in report["notices"])
    else:
        warning = ""
    if comps:
        intel_text = f"""
        <div class="panel">
          <div class="section-title">What the uploaded report says</div>
          <p class="subtle">Parsed {len(comps)} scenario row(s) from consolidated comparisons. Attack names come from the report. Explanations below are rule-based until Gemini is invoked.</p>
          {''.join(f'<p><b>{html.escape(r["Attack"])}</b> — {html.escape(explanations[i])}</p>' for i,r in enumerate(comps))}
          <p class="subtle">Metric guardrail: the Noxim ideal-flit ratio can exceed 100%; it is not a probability. Never label its change as an attack-risk reduction without a defined metric.</p>
          {f'<p class="status-warn">{warning}</p>' if warning else ''}
        </div>
        """
    else:
        intel_text = f"""
        <div class="panel"><div class="section-title">Sample intelligence</div>
        <p class="subtle">No consolidated comparison table was detected. The uploaded scenario CSVs are shown individually using filename / embedded scenario labels and available aggregate metrics. Generic names such as attack_102 cannot identify a precise attack type alone.</p>
        {f'<p class="status-warn">{warning}</p>' if warning else ''}
        </div>
        """
    chart1 = metric_chart(report, defense=False)
    chart2 = metric_chart(report, defense=True)
    path_v = n_verified > 0 and n_verified == len(comps) if comps else n_verified > 0
    delivery_v = n_delivery > 0 and n_delivery == len(comps) if comps else n_delivery > 0
    topology = svg_topology(target if target is not None else 6, 7, path_v, delivery_v)
    defense_text = f"""
    <div class="panel">
      <div class="section-title">Defense evidence</div>
      <p>{"<span class='status-ok'>Explicit path evidence found.</span>" if path_v else "<span class='status-warn'>PATH NOT VERIFIED:</span> no explicit route-proof marker in the uploaded evidence."}</p>
      <p>{"<span class='status-ok'>Explicit destination-delivery evidence found.</span>" if delivery_v else "<span class='status-warn'>FAILOVER DELIVERY NOT VERIFIED:</span> a mitigation hook alone is insufficient."}</p>
      <p class="subtle">For your demo, upload the comparison CSV plus matching Noxim run logs if available. This lets the dashboard separate scenario metrics from evidence about whether protection actually worked.</p>
      {f'<p class="status-warn">{warning}</p>' if warning else ''}
    </div>
    """
    # Compact LLM-ready evidence: values only from the uploaded data, never generated percentages.
    ai_summary = {
        "files": [Path(p).name for p in paths],
        "target_node": target,
        "comparisons": comps[:12],
        "samples": samples[:20],
        "log_evidence": evidence[:20],
        "verification": {"mitigation_hook_rows": n_hooks, "path_verified_rows": n_verified, "delivery_verified_rows": n_delivery}
    }
    state = json.dumps(ai_summary, default=str)[:24000]
    return intel_kpis, intel_text, table, chart1, def_kpis, topology, chart2, defense_text, ev_df, state

def gemini_explanation(api_key, state):
    if not state:
        return "Upload simulation evidence and click Analyse first."
    if not api_key or not api_key.strip():
        return ("Enter your Gemini API key to request a generated analysis. "
                "Without a key, the dashboard still shows evidence-based rule explanations. "
                "Do not paste the key into a public report or screenshot.")
    prompt = f"""You are a cautious network-on-chip security analyst. Explain the uploaded Noxim evidence to an engineering student.
Only use supplied data. Do not invent attack percentages, causes, route histories, successful mitigation, or packet-delivery claims. Distinguish:
1) named scenario / attack behaviour,
2) measured metric changes,
3) mitigation-hook evidence,
4) verified path evidence,
5) verified destination delivery.
The Noxim ideal-flit ratio metric may exceed 100% and is not a probability. Explain in plain language why each named attack matters, what the data supports, what it does not support, and the next test needed. Use concise headings and provide a brief viva-ready summary.
Uploaded data JSON:
{state}"""
    body = json.dumps({"contents":[{"parts":[{"text":prompt}]}],
                       "generationConfig":{"temperature":0.2,"maxOutputTokens":850}}).encode("utf-8")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent?key={api_key.strip()}"
    req = urllib.request.Request(url, data=body, headers={"Content-Type":"application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=45) as response:
            data = json.loads(response.read().decode("utf-8"))
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="ignore")[:1000]
        return f"Gemini API returned HTTP {e.code}. Check that your API key and model access are valid. Details: {detail}"
    except Exception as e:
        return f"Gemini request failed: {type(e).__name__}: {e}"

APP_THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.cyan,
    secondary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.slate
)

with gr.Blocks(title="NoC Sentinel | Network Security Intelligence") as demo:
    state = gr.State("")
    gr.HTML("""
    <div class="hero">
      <div class="eyebrow">Network security · Noxim simulation intelligence</div>
      <h1>◈ NoC Sentinel</h1>
      <p>Attack intelligence and defense recovery — evidence first, metrics explained, routes never faked.</p>
    </div>
    """)
    with gr.Row():
        upload_files = gr.File(
            label="Upload simulation evidence · select multiple CSV / LOG / TXT files, or one ZIP",
            file_count="multiple", file_types=[".csv",".log",".txt",".zip"], type="filepath",
            scale=5
        )
        with gr.Column(scale=2):
            gr.Markdown("**Workflow**")
            gr.Markdown("1. Select 1–25 scenario files and logs\n2. Click Analyse once\n3. Review both dashboards")
            analyze_btn = gr.Button("Analyse uploaded evidence", variant="primary", size="lg")
    with gr.Tabs():
        with gr.Tab("01 · Attack Intelligence"):
            gr.Markdown("### Scenario classification · evidence review")
            intel_kpis = gr.HTML()
            with gr.Row():
                with gr.Column(scale=5):
                    intel_chart = gr.Plot(label="Attack scenarios / measurements")
                with gr.Column(scale=4):
                    intel_summary = gr.HTML()
            intel_table = gr.Dataframe(label="Scenario and attack evidence", interactive=False, wrap=True)
            gr.Markdown("### Gemini security analyst")
            gr.Markdown("Optional: add a Google Gemini API key to explain the observed data. The key is used for this request and is not written to a file.")
            with gr.Row():
                gemini_key = gr.Textbox(label="Gemini API key", type="password", placeholder="Paste your key here; leave blank to use rule explanations")
                gemini_btn = gr.Button("Explain findings with Gemini", variant="secondary")
            ai_output = gr.Markdown("Gemini explanation will appear here. Without an API key, the dashboard still shows grounded rule-based explanations.")
        with gr.Tab("02 · Defense & Recovery"):
            gr.Markdown("### Mitigation effectiveness · failover topology")
            defense_kpis = gr.HTML()
            topology_html = gr.HTML()
            defense_chart = gr.Plot(label="Mitigation comparison")
            defense_summary = gr.HTML()
            evidence_table = gr.Dataframe(label="Log and route evidence", interactive=False, wrap=True)
            gr.Markdown("**Verification rule:** a `MITIGATION` log line indicates a mitigation hook ran. It does not independently prove that Node 6 was isolated, that Node 7 received traffic, or that the correct destination received the packet.")

    analyze_btn.click(
        analyze_files, inputs=[upload_files],
        outputs=[intel_kpis,intel_summary,intel_table,intel_chart,
                 defense_kpis,topology_html,defense_chart,defense_summary,evidence_table,state]
    )
    upload_files.change(
        analyze_files, inputs=[upload_files],
        outputs=[intel_kpis,intel_summary,intel_table,intel_chart,
                 defense_kpis,topology_html,defense_chart,defense_summary,evidence_table,state]
    )
    gemini_btn.click(gemini_explanation, inputs=[gemini_key,state], outputs=[ai_output])

demo.queue(default_concurrency_limit=4).launch(share=True, show_error=True, theme=APP_THEME, css=CSS)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8f526750b510821f2e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
